# ThriveLens Food-R1 — production-grade dual-GPU fine-tuning

This notebook performs fixed-epoch supervised fine-tuning of the public `zy12123/Food-R1` vision-language model on the verified 302,157-record ThriveLens worldwide food dataset. It is designed specifically for **Kaggle GPU T4 ×2** and is intentionally verbose: every executable cell announces its work, reports timing, prints its outputs, and prints a full traceback before re-raising an error.

The training architecture uses **4-bit NormalFloat (NF4) Quantized Low-Rank Adaptation (QLoRA)**, FP16 compute, DeepSpeed ZeRO Stage 2, Distributed Data Parallel (DDP), gradient checkpointing, a paged 8-bit AdamW optimizer, gradient clipping, cosine decay, warm-up, weight decay, group-safe validation, periodic recoverable checkpoints, and deterministic held-out generation. There is **no early stopping**, no hidden experiment tracker, and no account sign-in requirement for the public model or data. The run is scoped to one epoch with recoverable checkpoints so it fits within Kaggle's weekly GPU quota and per-session limits. An optional `HF_TOKEN` secret activates a private Hugging Face Hub auto-sync, making cross-session checkpoint restore fully automatic.

Two-T4 constraint: QLoRA and trainable visual layers are not a safe combination in this trainer. Food-R1's food-specialized vision tower is therefore frozen; LoRA adapts every eligible language-path linear layer. This preserves the pretrained food representation while fitting worldwide classification, ingredient, and nutrition supervision within 16 GiB per GPU.

Before running: attach the ThriveLens dataset to a Kaggle notebook, select **GPU T4 ×2**, enable Internet for public package/model downloads, then choose **Run all**. The notebook handles either the original ZIP or Kaggle's automatically extracted form without duplicating the image payload.

In [1]:
# pyright: ignore
# 01 — Runtime framework: visible stages, cleanup-safe subprocesses, telemetry, and tracebacks.
from __future__ import annotations

import contextlib
import datetime as dt
import hashlib
import io
import json
import math
import os
import platform
import shutil
import signal
import subprocess
import sys
import threading
import time
import traceback
import zipfile
from collections import Counter, defaultdict
from pathlib import Path


@contextlib.contextmanager
def stage(number: str, title: str):
    started = time.perf_counter()
    print("\n" + "=" * 100, flush=True)
    print(f"[STAGE {number}] {title}", flush=True)
    print(f"[START] {dt.datetime.now(dt.timezone.utc).isoformat()}", flush=True)
    print("=" * 100, flush=True)
    try:
        yield
    except Exception as exc:
        print(f"\n[STAGE {number} FAILED] {type(exc).__name__}: {exc}", file=sys.stderr, flush=True)
        traceback.print_exc()
        raise
    else:
        print(f"[STAGE {number} OK] elapsed={time.perf_counter() - started:.2f}s", flush=True)


def print_json(value):
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str), flush=True)


def sha256_file(path: Path, progress_gib: float = 0.5) -> str:
    digest = hashlib.sha256()
    processed = 0
    next_report = int(progress_gib * 2**30)
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(8 * 1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
            processed += len(chunk)
            if processed >= next_report:
                print(f"[SHA-256] processed {processed / 2**30:.2f} GiB", flush=True)
                next_report += int(progress_gib * 2**30)
    return digest.hexdigest()


def _terminate_process_tree(process: subprocess.Popen) -> None:
    if process.poll() is not None:
        return
    killpg = getattr(os, "killpg", None)
    sigterm = getattr(signal, "SIGTERM", None) or 15
    sigkill = getattr(signal, "SIGKILL", None) or sigterm
    try:
        if killpg is not None:
            killpg(process.pid, sigterm)
        else:
            process.terminate()
        process.wait(timeout=30)
    except Exception:
        try:
            if killpg is not None:
                killpg(process.pid, sigkill)
            else:
                process.kill()
            process.wait(timeout=10)
        except Exception:
            pass


def run_streaming(command, *, cwd=None, env=None, log_path=None, label="PROCESS"):
    command = [str(part) for part in command]
    print(f"[{label}] cwd={cwd or Path.cwd()}", flush=True)
    print(f"[{label}] command:", flush=True)
    print(" ".join(command), flush=True)
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
    popen_kwargs = {
        "cwd": str(cwd) if cwd else None,
        "env": env,
        "stdout": subprocess.PIPE,
        "stderr": subprocess.STDOUT,
        "text": True,
        "encoding": "utf-8",
        "errors": "replace",
        "bufsize": 1,
    }
    if hasattr(os, "setsid"):
        popen_kwargs["start_new_session"] = True
    process = subprocess.Popen(command, **popen_kwargs)
    tail = []
    try:
        log_handle = log_path.open("a", encoding="utf-8") if log_path else contextlib.nullcontext()
        with log_handle as log:
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end="", flush=True)
                if log:
                    log.write(line)
                    log.flush()
                tail.append(line.rstrip())
                if len(tail) > 120:
                    tail.pop(0)
        return_code = process.wait()
        if return_code != 0:
            print(f"[{label} ERROR] exit_code={return_code}", file=sys.stderr, flush=True)
            print("\n".join(tail[-80:]), file=sys.stderr, flush=True)
            raise subprocess.CalledProcessError(return_code, command)
        print(f"[{label}] completed successfully with exit_code=0", flush=True)
        return return_code
    except BaseException:
        print(f"[{label}] terminating child process tree after interruption/failure", file=sys.stderr, flush=True)
        _terminate_process_tree(process)
        raise


@contextlib.contextmanager
def live_telemetry(interval_seconds=60):
    stop = threading.Event()

    def monitor():
        while not stop.is_set():
            stamp = dt.datetime.now(dt.timezone.utc).strftime("%H:%M:%S UTC")
            try:
                gpu = subprocess.run(
                    ["nvidia-smi", "--query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw", "--format=csv,noheader,nounits"],
                    text=True, capture_output=True, check=True,
                ).stdout.strip().replace("\n", " | ")
            except Exception as exc:
                gpu = f"nvidia-smi error: {exc}"
            try:
                import psutil
                cpu = psutil.cpu_percent(interval=None)
                memory = psutil.virtual_memory().percent
                disk = psutil.disk_usage("/kaggle/working").percent
                host = f"CPU={cpu:.1f}% RAM={memory:.1f}% WORK_DISK={disk:.1f}%"
            except Exception as exc:
                host = f"host telemetry error: {exc}"
            print(f"[TELEMETRY {stamp}] {host} | GPUs: {gpu}", flush=True)
            stop.wait(interval_seconds)

    thread = threading.Thread(target=monitor, name="thrivelens-telemetry", daemon=True)
    thread.start()
    try:
        yield
    finally:
        stop.set()
        thread.join(timeout=5)
        print("[TELEMETRY] monitor stopped", flush=True)


with stage("01", "Initialize the transparent execution framework"):
    print(f"Python executable: {sys.executable}")
    print(f"Python version: {sys.version}")
    print(f"Platform: {platform.platform()}")
    print("Subprocesses run in a cleanup-safe process group; logs are streamed and persisted.")





[STAGE 01] Initialize the transparent execution framework
[START] 2026-09-08T19:00:39.618344+00:00
Python executable: /usr/bin/python3
Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Subprocesses run in a cleanup-safe process group; logs are streamed and persisted.
[STAGE 01 OK] elapsed=0.00s


In [2]:
# pyright: ignore
# 02 — Research-backed hyperparameters and a single source of truth for every downstream stage.
with stage("02", "Declare and validate the complete experiment contract"):
    MODEL_ID = "zy12123/Food-R1"
    EPOCHS = 1
    EARLY_STOPPING = False

    MICRO_BATCH = 1
    PER_DEVICE_EVAL_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8
    WORLD_SIZE = 2
    GLOBAL_BATCH_SIZE = MICRO_BATCH * GRADIENT_ACCUMULATION_STEPS * WORLD_SIZE

    LEARNING_RATE = 2e-4
    LR_SCHEDULER_TYPE = "cosine"
    WARMUP_RATIO = 0.05
    WEIGHT_DECAY = 0.01
    MAX_GRAD_NORM = 0.3
    OPTIMIZER = "paged_adamw_8bit"
    ADAM_BETA1 = 0.9
    ADAM_BETA2 = 0.999
    ADAM_EPSILON = 1e-8

    LOAD_IN_4BIT = True
    BNB_4BIT_QUANT_TYPE = "nf4"
    BNB_4BIT_USE_DOUBLE_QUANT = True
    BNB_4BIT_COMPUTE_DTYPE = "float16"
    LORA_RANK = 64
    LORA_ALPHA = 16
    LORA_DROPOUT = 0.05
    LORA_TARGET_MODULES = "all-linear"
    LORA_EXCLUDED_NAME_SPANS = ("lm_head", "embed_tokens", "visual")
    LORA_BIAS = "none"
    LORA_ENABLE = True
    VISION_LORA = False
    USE_DORA = False
    FREEZE_LLM = True
    FREEZE_VISION_TOWER = True
    FREEZE_MERGER = True
    UNFREEZE_TOPK_LLM = 0
    UNFREEZE_TOPK_VISION = 0

    FP16 = True
    BF16 = False
    TF32 = False
    GRADIENT_CHECKPOINTING = True
    GRADIENT_CHECKPOINTING_KWARGS = {"use_reentrant": False}
    DDP_FIND_UNUSED_PARAMETERS = False
    DDP_TIMEOUT_SECONDS = 7_200
    DISABLE_FLASH_ATTENTION_2 = True
    # T4 (Turing SM75) cannot build/run Flash Attention 2 (requires SM80+); torch SDPA is the only working fast path.
    USE_LIGER_KERNEL = False
    REMOVE_UNUSED_COLUMNS = False
    FULL_DETERMINISM = False

    MAX_SEQUENCE_LENGTH = 384
    MIN_IMAGE_PIXELS = 256 * 256
    MAX_IMAGE_PIXELS = 384 * 384
    TRAIN_ROWS_EXPECTED = 242_756
    VAL_ROWS_EXPECTED = 6_460
    TEST_ROWS_EXPECTED = 52_941

    EVAL_STRATEGY = "epoch"
    SAVE_STRATEGY = "steps"
    SAVE_STEPS = 1_000
    SAVE_TOTAL_LIMIT = 4
    LOGGING_STRATEGY = "steps"
    LOGGING_STEPS = 10
    LOG_FIRST_STEP = True
    LOGGING_NAN_INF_FILTER = False
    DISABLE_TQDM = False
    PREDICTION_LOSS_ONLY = True
    SAVE_SAFETENSORS = True
    LOAD_BEST_MODEL_AT_END = False
    DATALOADER_WORKERS = 4
    DATALOADER_PREFETCH_FACTOR = 2
    DATALOADER_PIN_MEMORY = True
    DATALOADER_PERSISTENT_WORKERS = True

    RUN_DUAL_GPU_SMOKE_TEST = True
    SMOKE_STEPS = 2
    SMOKE_EXAMPLES_PER_TASK = 8
    MAX_TEST_SAMPLES = 0          # Zero evaluates all 52,941 untouched test records.
    MAX_NEW_TOKENS = 256
    GENERATION_DO_SAMPLE = False
    GENERATION_TEMPERATURE = 0.0
    GENERATION_NUM_BEAMS = 1
    TELEMETRY_SECONDS = 60
    SEED = 2026
    MAX_TRAINING_GPU_HOURS = 8.0

    AUTO_SYNC = True
    RESUME_REPO_SHORT = "thrivelens-foodr1-resume"
    SYNC_POLL_SECONDS = 20
    SYNC_SETTLE_SECONDS = 30
    SYNC_REMOTE_HISTORY = 3

    EXPECTED_ARCHIVE_BYTES = 6_844_486_847
    EXPECTED_ZIP_ENTRIES = 302_249
    EXPECTED_ARCHIVE_SHA256 = "4c378fc53d427fff808a2bfee7d923d5569355fd3794b5ffd5bfc3a4115d7f49"

    WORK = Path("/kaggle/working/thrivelens_setup")
    OUTPUT_DIR = Path("/kaggle/working/thrivelens_foodr1_qlora")
    HF_HOME = Path("/kaggle/working/huggingface_cache")
    WORK.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    HF_HOME.mkdir(parents=True, exist_ok=True)

    os.environ.update({
        "HF_HOME": str(HF_HOME),
        "HUGGINGFACE_HUB_CACHE": str(HF_HOME / "hub"),
        "TRANSFORMERS_CACHE": str(HF_HOME / "hub"),
        "TOKENIZERS_PARALLELISM": "false",
        "WANDB_DISABLED": "true",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
        "CUDA_VISIBLE_DEVICES": ",".join(str(index) for index in range(WORLD_SIZE)),
        "TORCH_NCCL_ASYNC_ERROR_HANDLING": "1",
        "TORCH_DISTRIBUTED_DEBUG": "INFO",
    })

    approximate_steps = math.ceil(TRAIN_ROWS_EXPECTED / GLOBAL_BATCH_SIZE) * EPOCHS
    config_report = {
        "model": MODEL_ID,
        "method": "NormalFloat 4-bit Quantized Low-Rank Adaptation supervised fine-tuning",
        "epochs": EPOCHS,
        "early_stopping": EARLY_STOPPING,
        "batches": {"micro_per_gpu": MICRO_BATCH, "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS, "world_size": WORLD_SIZE, "global_effective": GLOBAL_BATCH_SIZE},
        "optimizer": {"name": OPTIMIZER, "learning_rate": LEARNING_RATE, "scheduler": LR_SCHEDULER_TYPE, "warmup_ratio": WARMUP_RATIO, "weight_decay": WEIGHT_DECAY, "max_grad_norm": MAX_GRAD_NORM},
        "quantization": {"load_in_4bit": LOAD_IN_4BIT, "type": BNB_4BIT_QUANT_TYPE, "double_quant": BNB_4BIT_USE_DOUBLE_QUANT, "compute_dtype": BNB_4BIT_COMPUTE_DTYPE},
        "LoRA": {"enabled": LORA_ENABLE, "rank": LORA_RANK, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGET_MODULES, "excluded_name_spans": LORA_EXCLUDED_NAME_SPANS, "bias": LORA_BIAS, "vision_lora": VISION_LORA, "use_dora": USE_DORA},
        "precision": {"fp16": FP16, "bf16": BF16, "tf32": TF32},
        "attention": {"implementation": "torch scaled_dot_product_attention" if DISABLE_FLASH_ATTENTION_2 else "flash_attention_2", "device": "T4 (SM75) cannot run Flash Attention 2 (requires Ampere SM80+)"},
        "gradient_checkpointing": {"enabled": GRADIENT_CHECKPOINTING, "kwargs": GRADIENT_CHECKPOINTING_KWARGS},
        "sequence_length": MAX_SEQUENCE_LENGTH,
        "image_pixel_range": [MIN_IMAGE_PIXELS, MAX_IMAGE_PIXELS],
        "approximate_optimizer_steps": approximate_steps,
        "checkpointing": {"strategy": SAVE_STRATEGY, "every_optimizer_steps": SAVE_STEPS, "retained": SAVE_TOTAL_LIMIT},
        "validation_strategy": EVAL_STRATEGY,
        "held_out_generation_samples": MAX_TEST_SAMPLES or TEST_ROWS_EXPECTED,
        "max_training_gpu_hours": MAX_TRAINING_GPU_HOURS,
        "generation": {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": GENERATION_DO_SAMPLE, "temperature": GENERATION_TEMPERATURE, "num_beams": GENERATION_NUM_BEAMS},
        "auto_resume_sync": {"enabled": AUTO_SYNC, "repo": RESUME_REPO_SHORT, "kept_remote_history": SYNC_REMOTE_HISTORY},
    }
    assert EPOCHS == 1 and EARLY_STOPPING is False
    assert WORLD_SIZE == 2 and GLOBAL_BATCH_SIZE == 16
    assert 0 < LEARNING_RATE <= 2e-4 and 0.0 < WARMUP_RATIO < 1.0
    assert LOAD_IN_4BIT and BNB_4BIT_QUANT_TYPE == "nf4" and BNB_4BIT_COMPUTE_DTYPE == "float16"
    assert LORA_RANK == 64 and LORA_ALPHA == 16 and LORA_TARGET_MODULES == "all-linear"
    assert LORA_ENABLE and not VISION_LORA and not USE_DORA
    assert FREEZE_LLM and FREEZE_VISION_TOWER and FREEZE_MERGER
    assert FP16 and not BF16 and not TF32
    assert DISABLE_FLASH_ATTENTION_2, "T4 (SM75) cannot run Flash Attention 2; SDPA is the only viable attention implementation."
    assert GRADIENT_CHECKPOINTING and GRADIENT_CHECKPOINTING_KWARGS == {"use_reentrant": False}
    assert MAX_SEQUENCE_LENGTH == 384 and MAX_IMAGE_PIXELS == 384 * 384
    assert MAX_TEST_SAMPLES == 0 and not GENERATION_DO_SAMPLE and GENERATION_NUM_BEAMS == 1
    assert not LOAD_BEST_MODEL_AT_END and not EARLY_STOPPING
    print_json(config_report)
    (OUTPUT_DIR / "requested_configuration.json").write_text(json.dumps(config_report, indent=2), encoding="utf-8")



[STAGE 02] Declare and validate the complete experiment contract
[START] 2026-09-08T19:00:39.652022+00:00
{
  "model": "zy12123/Food-R1",
  "method": "NormalFloat 4-bit Quantized Low-Rank Adaptation supervised fine-tuning",
  "epochs": 1,
  "early_stopping": false,
  "batches": {
    "micro_per_gpu": 1,
    "gradient_accumulation": 8,
    "world_size": 2,
    "global_effective": 16
  },
  "optimizer": {
    "name": "paged_adamw_8bit",
    "learning_rate": 0.0002,
    "scheduler": "cosine",
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "max_grad_norm": 0.3
  },
  "quantization": {
    "load_in_4bit": true,
    "type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "LoRA": {
    "enabled": true,
    "rank": 64,
    "alpha": 16,
    "dropout": 0.05,
    "targets": "all-linear",
    "excluded_name_spans": [
      "lm_head",
      "embed_tokens",
      "visual"
    ],
    "bias": "none",
    "vision_lora": false,
    "use_dora": false
  },
  "precision": {

In [3]:
# pyright: ignore
# 02c — Final safety overrides applied immediately before runtime discovery.
with stage("02c", "Enforce the 20 GiB Kaggle working-disk profile"):
    SAVE_STEPS = 250
    SAVE_TOTAL_LIMIT = 2
    DATALOADER_WORKERS = 2
    DATALOADER_PREFETCH_FACTOR = 1
    DATALOADER_PERSISTENT_WORKERS = False
    MAX_TEST_SAMPLES = 400
    MIN_FREE_WORK_GIB = 17.5
    DISK_ABORT_PERCENT = 92.0
    RAM_ABORT_PERCENT = 94.0
    os.environ["PIP_NO_CACHE_DIR"] = "1"
    os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
    free_work_gib = shutil.disk_usage("/kaggle/working").free / 2**30
    if free_work_gib < MIN_FREE_WORK_GIB:
        _work_root = Path("/kaggle/working")
        _stale_bases = [HF_HOME, WORK / "remote_resume_stage"]
        if any(_path.exists() for _path in _stale_bases):
            print(f"[DISK] Free working storage is {free_work_gib:.2f} GiB. Pruning the stale base-model cache for a clean re-run.")
        for _path in _stale_bases:
            shutil.rmtree(_path, ignore_errors=True)
        for _broken in _work_root.rglob("checkpoint-*.invalid"):
            shutil.rmtree(_broken, ignore_errors=True)
        free_work_gib = shutil.disk_usage("/kaggle/working").free / 2**30
    assert free_work_gib >= MIN_FREE_WORK_GIB, (
        f"At least {MIN_FREE_WORK_GIB:.1f} GiB free working storage is required; found {free_work_gib:.2f} GiB."
    )
    try:
        import psutil
        available_ram_gib = psutil.virtual_memory().available / 2**30
        assert available_ram_gib >= 8.0, f"At least 8 GiB host RAM must be available; found {available_ram_gib:.2f} GiB."
    except ModuleNotFoundError:
        print("[SAFETY WARNING] psutil will be installed by Cell 08 before training.")
    print_json({
        "kaggle_working_disk_budget_gib": 20,
        "free_working_gib_at_launch": round(free_work_gib, 2),
        "checkpoint_steps": SAVE_STEPS,
        "retained_checkpoints": SAVE_TOTAL_LIMIT,
        "dataloader_workers_per_rank": DATALOADER_WORKERS,
        "prefetch_factor": DATALOADER_PREFETCH_FACTOR,
        "persistent_workers": DATALOADER_PERSISTENT_WORKERS,
        "held_out_diagnostic_samples": MAX_TEST_SAMPLES,
        "pip_download_cache": False,
        "minimum_free_working_gib": MIN_FREE_WORK_GIB,
    })




[STAGE 02c] Enforce the 20 GiB Kaggle working-disk profile
[START] 2026-09-08T19:00:39.668850+00:00
{
  "kaggle_working_disk_budget_gib": 20,
  "free_working_gib_at_launch": 19.5,
  "checkpoint_steps": 250,
  "retained_checkpoints": 2,
  "dataloader_workers_per_rank": 2,
  "prefetch_factor": 1,
  "persistent_workers": false,
  "held_out_diagnostic_samples": 2000,
  "pip_download_cache": false,
  "minimum_free_working_gib": 17.5
}
[STAGE 02c OK] elapsed=0.00s


In [4]:
# pyright: ignore
# 02b — Kaggle 20 GiB storage profile; this cell intentionally repeats the final effective settings.
with stage("02b", "Confirm the final 20 GiB Kaggle safety envelope"):
    SAVE_STEPS = 250
    SAVE_TOTAL_LIMIT = 2
    DATALOADER_WORKERS = 2
    DATALOADER_PREFETCH_FACTOR = 1
    DATALOADER_PERSISTENT_WORKERS = False
    MAX_TEST_SAMPLES = 400
    MIN_FREE_WORK_GIB = 17.5
    DISK_ABORT_PERCENT = 92.0
    RAM_ABORT_PERCENT = 94.0
    os.environ["PIP_NO_CACHE_DIR"] = "1"
    free_work_gib = shutil.disk_usage("/kaggle/working").free / 2**30
    assert free_work_gib >= MIN_FREE_WORK_GIB, (
        f"At least {MIN_FREE_WORK_GIB:.1f} GiB free working storage is required; found {free_work_gib:.2f} GiB."
    )
    config_report["checkpointing"] = {"strategy": SAVE_STRATEGY, "every_optimizer_steps": SAVE_STEPS, "retained": SAVE_TOTAL_LIMIT}
    config_report["held_out_generation_samples"] = MAX_TEST_SAMPLES
    config_report["dataloader"] = {"workers": DATALOADER_WORKERS, "prefetch_factor": DATALOADER_PREFETCH_FACTOR, "persistent_workers": DATALOADER_PERSISTENT_WORKERS}
    (OUTPUT_DIR / "requested_configuration.json").write_text(json.dumps(config_report, indent=2), encoding="utf-8")
    print_json({
        "working_disk_budget_gib": 20,
        "free_working_gib": round(free_work_gib, 2),
        "checkpoint_steps": SAVE_STEPS,
        "retained_checkpoints": SAVE_TOTAL_LIMIT,
        "dataloader_workers": DATALOADER_WORKERS,
        "prefetch_factor": DATALOADER_PREFETCH_FACTOR,
        "persistent_workers": DATALOADER_PERSISTENT_WORKERS,
        "held_out_diagnostic_samples": MAX_TEST_SAMPLES,
    })



[STAGE 02b] Confirm the final 20 GiB Kaggle safety envelope
[START] 2026-09-08T19:00:39.684968+00:00
{
  "working_disk_budget_gib": 20,
  "free_working_gib": 19.5,
  "checkpoint_steps": 250,
  "retained_checkpoints": 2,
  "dataloader_workers": 2,
  "prefetch_factor": 1,
  "persistent_workers": false,
  "held_out_diagnostic_samples": 2000
}
[STAGE 02b OK] elapsed=0.00s


In [5]:
# pyright: ignore
# 03 — Kaggle host, storage, public Internet, CUDA, and exact dual-GPU checks.
with stage("03", "Fail fast on an unsuitable Kaggle runtime"):
    import urllib.request
    import torch

    print("[SYSTEM] nvidia-smi")
    subprocess.run(["nvidia-smi"], check=True)
    print("[SYSTEM] filesystem capacity")
    subprocess.run(["df", "-h", "/kaggle/input", "/kaggle/working"], check=True)
    print("[SYSTEM] host memory")
    subprocess.run(["free", "-h"], check=True)
    print("[SYSTEM] /kaggle/working size breakdown (largest first)")
    _work_entries = []
    for _item in Path("/kaggle/working").iterdir():
        if _item.is_dir():
            _item_size = sum(_file.stat().st_size for _file in _item.rglob("*") if _file.is_file())
        else:
            _item_size = _item.stat().st_size
        _work_entries.append((_item_size, _item))
    _work_entries.sort(key=lambda _entry: _entry[0], reverse=True)
    for _size_bytes, _item in _work_entries[:15]:
        print(f"[SYSTEM] {_size_bytes / 2**20:10.1f} MiB  {_item}")

    assert Path("/kaggle/input").exists(), "This notebook is intended to run on Kaggle."
    assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle Notebook options."
    assert torch.cuda.device_count() == WORLD_SIZE, (
        f"Select GPU T4 x2. PyTorch can see {torch.cuda.device_count()} GPU(s), but exactly {WORLD_SIZE} are required."
    )
    gpu_report = []
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        item = {
            "index": index,
            "name": props.name,
            "compute_capability": f"{props.major}.{props.minor}",
            "total_memory_gib": round(props.total_memory / 2**30, 2),
            "bfloat16_supported": torch.cuda.is_bf16_supported(),
        }
        assert props.total_memory >= 14 * 2**30, f"GPU {index} has insufficient memory: {item}"
        gpu_report.append(item)
    print_json(gpu_report)

    free_work_gib = shutil.disk_usage("/kaggle/working").free / 2**30
    assert free_work_gib >= 16, f"At least 16 GiB free working storage is required; found {free_work_gib:.2f} GiB."
    if free_work_gib < 25:
        print(f"[WARNING] Working storage is {free_work_gib:.2f} GiB. The newest {SAVE_TOTAL_LIMIT} checkpoints will be retained.")

    config_url = "https://huggingface.co/zy12123/Food-R1/resolve/main/config.json"
    try:
        with urllib.request.urlopen(config_url, timeout=30) as response:
            public_config_bytes = response.read(256)
        print(f"[NETWORK] Public Food-R1 endpoint reachable; received {len(public_config_bytes)} bytes. No sign-in token is used.")
    except Exception as exc:
        raise RuntimeError("Enable Internet in Kaggle settings. Food-R1 is public and requires no account sign-in.") from exc


[STAGE 03] Fail fast on an unsuitable Kaggle runtime
[START] 2026-09-08T19:00:39.703327+00:00
[SYSTEM] nvidia-smi
Tue Sep  8 19:00:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                        

In [6]:
# pyright: ignore
# 04 — Discover the data in raw-ZIP, Kaggle-extracted, or local development form, then stage only small support files.
with stage("04", "Discover and verify the ThriveLens dataset package"):
    INPUT_ROOT = Path("/kaggle/input")
    LOCAL_ARCHIVE = None
    if platform.system() == "Windows":
        LOCAL_ARCHIVE = Path(r"c:\Users\thaba\OneDrive\Documents\ThriveLens 2026\Training and Validation Model\ThriveLens_OpenFood_FoodR1_Kaggle_300K.zip")
    named_archives = sorted(INPUT_ROOT.rglob("ThriveLens_OpenFood_FoodR1_Kaggle_300K.zip"))
    if not named_archives and LOCAL_ARCHIVE is not None and LOCAL_ARCHIVE.exists():
        named_archives = [LOCAL_ARCHIVE]
    manifest_candidates = sorted(INPUT_ROOT.rglob("dataset_manifest.json"))
    valid_manifests = []
    for candidate in manifest_candidates:
        try:
            candidate_manifest = json.loads(candidate.read_text(encoding="utf-8"))
            if "ThriveLens Open Food 300K" in candidate_manifest.get("name", ""):
                valid_manifests.append(candidate)
        except Exception as exc:
            print(f"[DISCOVERY WARNING] Ignoring unreadable manifest {candidate}: {exc}")

    ARCHIVE = None
    DATASET_ROOT = None
    DATA_MODE = None
    archive_sha256 = None
    if len(named_archives) == 1:
        DATA_MODE = "archive"
        ARCHIVE = named_archives[0]
    elif len(named_archives) > 1:
        raise RuntimeError(f"Attach only one ThriveLens archive; found: {named_archives}")
    elif len(valid_manifests) == 1:
        DATA_MODE = "extracted"
        DATASET_ROOT = valid_manifests[0].parent
    elif len(valid_manifests) > 1:
        raise RuntimeError(f"Attach only one extracted ThriveLens dataset; found: {valid_manifests}")
    else:
        nearby = [str(path) for path in sorted(INPUT_ROOT.iterdir())]
        raise FileNotFoundError(
            "The ThriveLens dataset is not attached. Expected the named ZIP or dataset_manifest.json under /kaggle/input. "
            f"Top-level Kaggle inputs: {nearby}"
        )

    support_names = {"README.md", "dataset_manifest.json", "SHA256SUMS.txt"}
    def _archive_trainer_entries(zf):
        _prefix = next(
            (name.split("/src/train/train_sft.py", 1)[0] for name in zf.namelist()
             if name.endswith("/src/train/train_sft.py")),
            None,
        )
        if not _prefix:
            return None
        return _prefix, [name for name in zf.namelist() if name.startswith(_prefix + "/")]

    def _stage_trainer_from_archive(zf) -> bool:
        _found = _archive_trainer_entries(zf)
        if not _found:
            return False
        _prefix, _entries = _found
        for _position, name in enumerate(_entries, start=1):
            zf.extract(name, WORK)
            if _position % 50 == 0:
                print(f"[DATA] trainer entries staged {_position}/{len(_entries)}", flush=True)
        _target = WORK / "training" / "qwen-vl-finetune"
        _extracted_full = WORK / _prefix
        if _extracted_full.is_dir() and _extracted_full != _target:
            _target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(_extracted_full), str(_target))
        return _target.exists()

    def _find_extracted_trainer_root() -> Path | None:
        if DATASET_ROOT is None:
            return None
        for _probe in (DATASET_ROOT / "training" / "qwen-vl-finetune", DATASET_ROOT / "qwen-vl-finetune"):
            if (_probe / "src" / "train" / "train_sft.py").exists() and (_probe / "scripts" / "zero2.json").exists():
                return _probe
        _seen = set()
        for _hit in DATASET_ROOT.rglob("train_sft.py"):
            if _hit in _seen:
                continue
            _seen.add(_hit)
            for _parent in _hit.parents:
                if (_parent / "scripts" / "zero2.json").exists() and (_parent / "src" / "dataset" / "data_utils.py").exists():
                    return _parent
        return None

    if DATA_MODE == "archive":
        if not isinstance(ARCHIVE, Path):
            raise FileNotFoundError(f"DATA_MODE={DATA_MODE} requires a resolved ARCHIVE, got {ARCHIVE!r}")
        print(f"[DATA] Mode=raw ZIP | path={ARCHIVE} | size={ARCHIVE.stat().st_size / 2**30:.3f} GiB")
        assert ARCHIVE.stat().st_size == EXPECTED_ARCHIVE_BYTES, (
            f"Archive size mismatch: expected {EXPECTED_ARCHIVE_BYTES:,} bytes, received {ARCHIVE.stat().st_size:,}."
        )
        archive_sha256 = sha256_file(ARCHIVE)
        print(f"[DATA] SHA-256={archive_sha256}")
        assert archive_sha256 == EXPECTED_ARCHIVE_SHA256, (
            "Archive hash mismatch. The download is incomplete or this is not the verified ThriveLens 300K build. "
            f"Expected {EXPECTED_ARCHIVE_SHA256}, received {archive_sha256}."
        )
        with zipfile.ZipFile(ARCHIVE) as zf:
            bad_member = zf.testzip()
            assert bad_member is None, f"ZIP CRC validation failed at {bad_member}"
            names = zf.namelist()
            assert len(names) == EXPECTED_ZIP_ENTRIES, (
                f"ZIP entry mismatch: expected {EXPECTED_ZIP_ENTRIES:,}, received {len(names):,}."
            )
            wanted = [name for name in names if name.startswith("sft/") or name in support_names]
            print(f"[DATA] ZIP entries={len(names):,}; staging support entries={len(wanted):,}")
            for position, name in enumerate(wanted, start=1):
                zf.extract(name, WORK)
                if position % 10 == 0 or position == len(wanted):
                    print(f"[DATA] staged {position:,}/{len(wanted):,} support entries", flush=True)
            assert _stage_trainer_from_archive(zf), "Pinned trainer src/train/train_sft.py not found inside the attached archive"
            print("[DATA] pinned trainer qwen-vl-finetune staged under WORK/training", flush=True)
        os.environ["THRIVELENS_ARCHIVE"] = str(ARCHIVE)
        os.environ.pop("THRIVELENS_DATASET_ROOT", None)
    else:
        assert DATASET_ROOT is not None
        print(f"[DATA] Mode=Kaggle-extracted | root={DATASET_ROOT}")
        _training_source = DATASET_ROOT / "training"
        assert _training_source.exists(), f"Required directory missing: {_training_source}"
        print(f"[DATA] training media stay at {_training_source} and stream from there during training (no working-disk copy; saves ~21 min per session restart)")
        _trainer_source = _find_extracted_trainer_root()
        assert _trainer_source is not None, (
            "The pinned Qwen3-VL trainer tree (qwen-vl-finetune with src/train/train_sft.py and scripts/zero2.json) "
            "was not found under the extracted dataset."
        )
        _trainer_target = WORK / "training" / "qwen-vl-finetune"
        shutil.copytree(_trainer_source, _trainer_target, dirs_exist_ok=True)
        print(f"[DATA] pinned trainer copied {_trainer_source} -> {_trainer_target}", flush=True)
        for directory in ("sft",):
            source = DATASET_ROOT / directory
            assert source.exists(), f"Required directory missing: {source}"
            print(f"[DATA] staging {source} -> {WORK / directory}")
            shutil.copytree(source, WORK / directory, dirs_exist_ok=True)
        for name in support_names:
            source = DATASET_ROOT / name
            if source.exists():
                shutil.copy2(source, WORK / name)
        archive_sha256 = "not-applicable-kaggle-extracted-mode"
        os.environ["THRIVELENS_DATASET_ROOT"] = str(DATASET_ROOT)
        os.environ.pop("THRIVELENS_ARCHIVE", None)

    SFT_DIR = WORK / "sft"
    assert (WORK / "dataset_manifest.json").exists()          
    for split in ("train", "val", "test"):
        assert (SFT_DIR / f"{split}.json").exists(), f"Missing {split}.json"
        assert (SFT_DIR / f"{split}.jsonl").exists(), f"Missing {split}.jsonl"
    print_json({
        "mode": DATA_MODE,
        "archive": str(ARCHIVE) if ARCHIVE else None,
        "extracted_root": str(DATASET_ROOT) if DATASET_ROOT else None,
        "support_workspace": str(WORK),
        "archive_sha256": archive_sha256,
    })
def iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                yield line_number, json.loads(line)



[STAGE 04] Discover and verify the ThriveLens dataset package
[START] 2026-09-08T19:00:44.249800+00:00
[DATA] Mode=Kaggle-extracted | root=/kaggle/input/datasets/neosetlolela/thrivelensedata
[DATA] staging /kaggle/input/datasets/neosetlolela/thrivelensedata/training -> /kaggle/working/thrivelens_setup/training
[DATA] staging /kaggle/input/datasets/neosetlolela/thrivelensedata/sft -> /kaggle/working/thrivelens_setup/sft
{
  "mode": "extracted",
  "archive": null,
  "extracted_root": "/kaggle/input/datasets/neosetlolela/thrivelensedata",
  "support_workspace": "/kaggle/working/thrivelens_setup",
  "archive_sha256": "not-applicable-kaggle-extracted-mode"
}
[STAGE 04 OK] elapsed=1286.36s


In [7]:
# pyright: ignore
# 05 — Exact source, split, and task inventory from the signed package manifest.
with stage("05", "Inspect the worldwide dataset composition"):
    manifest = json.loads((WORK / "dataset_manifest.json").read_text(encoding="utf-8"))  
    summary = manifest["summary"]
    expected_split_counts = {"train": TRAIN_ROWS_EXPECTED, "val": VAL_ROWS_EXPECTED, "test": TEST_ROWS_EXPECTED}
    assert summary["sft_records"] == expected_split_counts, (summary["sft_records"], expected_split_counts)
    assert summary["open_images_food_crops"] == 107_794
    assert summary["fruits360_images"] == 183_893
    assert summary["nutrition5k_supervised_dishes"] == 3_490

    print("[DATASET] verified sources and licenses")
    for source in manifest["sources"]:
        print_json(source)
    print("[DATASET] split/task inventory")
    print_json({"records": summary["sft_records"], "tasks": summary["sft_tasks"]})
    print("[DATASET] split policy")
    print(manifest["split_policy"])



[STAGE 05] Inspect the worldwide dataset composition
[START] 2026-09-08T19:22:10.617673+00:00
[DATASET] verified sources and licenses
{
  "name": "Nutrition5k",
  "license": "CC BY 4.0",
  "url": "https://github.com/google-research-datasets/Nutrition5k",
  "included": "all official RealSense overhead RGB, raw depth, colorized depth, metadata, dish IDs/splits, scripts"
}
{
  "name": "Open Images",
  "image_license": "CC BY 2.0",
  "annotation_license": "CC BY 4.0",
  "url": "https://storage.googleapis.com/openimages/web/download_v7.html",
  "included": "clean food bounding-box crops with per-image attribution"
}
{
  "name": "Fruits-360 100x100",
  "url": "https://github.com/fruits-360/fruits-360-100x100",
  "included": "all official Training and Test images across fruits, vegetables, nuts, seeds and fungi",
  "upstream_license_files": [
    "source_tables/fruits360/LICENSE",
    "source_tables/fruits360/README.md"
  ]
}
[DATASET] split/task inventory
{
  "records": {
    "train": 24275

## Software and trainer verification

The next cells install pinned versions with fully streamed output, verify CUDA-aware libraries, stage the commit-pinned Qwen-VL trainer, and apply an idempotent media resolver. The resolver opens each ZIP independently per DataLoader process, avoiding inherited file handles, and also supports Kaggle-extracted files.


In [10]:
# pyright: ignore
# 08 — Install the pinned training stack with no pip cache.
# Kaggle-safe: packages install one at a time, DeepSpeed build failures degrade to a
# clear warning instead of aborting the notebook, and importability is checked live.
with stage("08", "Install and record the deep-learning software stack"):
    def _installed_version(package_name: str):
        try:
            module = __import__(package_name)
            return getattr(module, "__version__", "present")
        except Exception:
            return None

    def _imports_package(package_name: str) -> bool:
        try:
            __import__(package_name)
            return True
        except Exception:
            return False

    def _pip_install(args: list) -> bool:
        command = [
            sys.executable, "-m", "pip", "install", "--upgrade", "--no-input",
            "--no-cache-dir", "--progress-bar", "off", *args,
        ]
        log_path = WORK / "pip_install.log"
        try:
            run_streaming(command, log_path=log_path, label="PIP")
            return True
        except BaseException:
            return False

    mandate = [
        "transformers==5.3.0",
        "accelerate==1.10.1",
        "peft==0.15.2",
        "bitsandbytes==0.49.2",
        "trl==0.25.0",
        "qwen-vl-utils",
        "ujson==5.10.0",
        "tensorboardX==2.6.2.2",
        "ninja",
        "safetensors",
        "psutil",
        "pandas",
        "matplotlib",
        "tabulate",
    ]
    deepspeed_spec = "deepspeed==0.17.5"
    print("[PACKAGES] requested pins:")
    print("\n".join(f"  - {package}" for package in mandate + [deepspeed_spec]))
    print("[PACKAGES] Installing each package separately so a single build failure cannot abort the run.")
    installed = {}
    failed = []
    for spec in mandate:
        name = spec.split("==", 1)[0]
        pinned = spec.split("==", 1)[1] if "==" in spec else None
        current = _installed_version(name)
        if _imports_package(name) and (pinned is None or current == pinned):
            print(f"[PACKAGES] {name} already at {current}; skipping reinstall.")
            installed[name] = current
            continue
        if _pip_install([spec]):
            installed[name] = _installed_version(name)
        else:
            failed.append(spec)
            print(f"[PACKAGES WARNING] {spec} failed to install; continuing so the notebook stays runnable.", file=sys.stderr, flush=True)

    print("[PACKAGES] DeepSpeed install (this is the step that occasionally times out on Kaggle).")
    if _imports_package("deepspeed"):
        print(f"[PACKAGES] deepspeed already importable ({_installed_version('deepspeed')}); skipping build.")
        installed["deepspeed"] = _installed_version("deepspeed")
    else:
        ds_ok = _pip_install(["--no-build-isolation", deepspeed_spec]) or _pip_install([deepspeed_spec])
        if ds_ok and _imports_package("deepspeed"):
            installed["deepspeed"] = _installed_version("deepspeed")
        else:
            failed.append("deepspeed")
            print(
                "[PACKAGES WARNING] DeepSpeed could not be installed or imported on this runtime. "
                "DeepSpeed-based training will be unavailable; re-check the compile toolchain or Kaggle image. ",
                file=sys.stderr, flush=True,
            )

    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], text=True, capture_output=True).stdout
    (OUTPUT_DIR / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
    print("[PACKAGES] relevant installed versions:")
    for line in freeze.splitlines():
        key = line.lower().split("==")[0]
        if key in {"transformers", "accelerate", "peft", "bitsandbytes", "deepspeed", "trl", "torch", "qwen-vl-utils"}:
            print("  " + line)
    hard_requirements = {"transformers", "accelerate", "peft", "bitsandbytes", "trl", "torch"}
    missing_hard = [name for name in hard_requirements if name not in installed and not _imports_package(name)]
    if missing_hard:
        raise RuntimeError(
            f"Essential packages could not be installed: {missing_hard}. "
            "Check the Kaggle environment/Internet and re-run this cell or the whole notebook."
        )
    print_json({"installed": installed, "failed_but_non_fatal": failed})



[STAGE 08] Install and record the deep-learning software stack
[START] 2026-09-08T19:28:41.048304+00:00
[PACKAGES] requested pins:
  - transformers==5.3.0
  - accelerate==1.10.1
  - peft==0.15.2
  - bitsandbytes==0.49.2
  - trl==0.25.0
  - qwen-vl-utils
  - ujson==5.10.0
  - tensorboardX==2.6.2.2
  - ninja
  - safetensors
  - psutil
  - pandas
  - matplotlib
  - tabulate
  - deepspeed==0.17.5
[PACKAGES] Installing each package separately so a single build failure cannot abort the run.
[PIP] cwd=/kaggle/working
[PIP] command:
/usr/bin/python3 -m pip install --upgrade --no-input --no-cache-dir --progress-bar off transformers==5.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
[PIP] completed successfully with exit_code=0
[PIP] cwd=/kaggle/working
[PIP] command:
/usr/bin/python3 -m pip install --upgrade --no-input --no-cache-dir --progress-bar off accelerate=

/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2026-09-08 19:30:27,605] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False
[PACKAGES] relevant installed versions:
  accelerate==1.10.1
  bitsandbytes==0.49.2
  deepspeed==0.17.5
  peft==0.15.2
  qwen-vl-utils==0.0.14
  torch==2.10.0+cu128
  transformers==5.3.0
  trl==0.25.0
{
  "installed": {
    "transformers": "5.0.0",
    "accelerate": "1.13.0",
    "peft": null,
    "bitsandbytes": "0.49.2",
    "trl": "0.25.0",
    "qwen-vl-utils": null,
    "ujson": "5.13.0",
    "tensorboardX": "2.6.2.2",
    "ninja": "1.13.0",
    "safetensors": "0.7.0",
    "psutil": "5.9.5",
    "pandas": "2.3.3",
    "matplotlib": "3.10.0",
    "tabulate": "0.9.0",
    "deepspeed": "0.17.5"
  },
  "failed_but_non_fatal": []
}
[STAGE 08 OK] elapsed=108.24s


In [11]:
# pyright: ignore
# 09 — Import-time compatibility, CUDA quantization, and DeepSpeed diagnostics.
with stage("09", "Verify every critical package and CUDA integration"):
    import transformers.utils.import_utils as _iu
    if not hasattr(_iu, "is_fouroversix_available"):
        _iu.is_fouroversix_available = lambda: False
        print("[PATCH] injected missing is_fouroversix_available into transformers.utils.import_utils")
    import accelerate
    import bitsandbytes as bnb
    import deepspeed
    import peft
    import safetensors
    import transformers
    import trl
    import torch

    version_report = {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torch_cuda_build": torch.version.cuda,
        "transformers": transformers.__version__,
        "accelerate": accelerate.__version__,
        "peft": peft.__version__,
        "bitsandbytes": bnb.__version__,
        "deepspeed": deepspeed.__version__,
        "trl": trl.__version__,
        "safetensors": safetensors.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_devices": torch.cuda.device_count(),
    }
    assert transformers.__version__ == "5.3.0"
    assert torch.cuda.device_count() == WORLD_SIZE
    print_json(version_report)
    (OUTPUT_DIR / "software_versions.json").write_text(json.dumps(version_report, indent=2), encoding="utf-8")

    print("[BITSANDBYTES] CUDA diagnostic")
    bnb_check = subprocess.run([sys.executable, "-m", "bitsandbytes"], text=True, capture_output=True)
    print(bnb_check.stdout)
    print(bnb_check.stderr)
    if bnb_check.returncode != 0:
        print("[BITSANDBYTES WARNING] CUDA diagnostics returned a non-zero exit code; continuing with a clear warning.", file=sys.stderr, flush=True)
    print("[DEEPSPEED] environment report")
    try:
        ds_check = subprocess.run(["ds_report"], text=True, capture_output=True, timeout=300)
        print(ds_check.stdout)
        print(ds_check.stderr)
        if ds_check.returncode != 0:
            print("[DEEPSPEED WARNING] environment report returned a non-zero exit code; DeepSpeed may be incomplete.", file=sys.stderr, flush=True)
    except Exception as exc:
        print(f"[DEEPSPEED WARNING] could not run ds_report ({type(exc).__name__}); DeepSpeed may be unavailable.", file=sys.stderr, flush=True)



[STAGE 09] Verify every critical package and CUDA integration
[START] 2026-09-08T19:30:29.302746+00:00
[PATCH] injected missing is_fouroversix_available into transformers.utils.import_utils
{
  "python": "3.12.13",
  "torch": "2.10.0+cu128",
  "torch_cuda_build": "12.8",
  "transformers": "5.3.0",
  "accelerate": "1.13.0",
  "peft": "0.15.2",
  "bitsandbytes": "0.49.2",
  "deepspeed": "0.17.5",
  "trl": "0.25.0",
  "safetensors": "0.7.0",
  "cuda_available": true,
  "cuda_devices": 2
}
[BITSANDBYTES] CUDA diagnostic
=================== bitsandbytes v0.49.2 ===================
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
  libc: glibc-2.35
Python: 3.12.13
PyTorch: 2.10.0+cu128
  CUDA: 12.8
  HIP: N/A
  XPU: N/A
Related packages:
  accelerate: 1.10.1
  diffusers: 0.37.1
  numpy: 2.0.2
  pip: 24.1.2
  peft: 0.15.2
  safetensors: 0.7.0
  transformers: 5.3.0
  triton: 3.6.0
  trl: 0.25.0
PyTorch settings found: CUDA_VERSION=128, Highest Compute Capability: (7, 5).
Checking that the libra

In [12]:
# pyright: ignore
# SYNC-AUX: optional automatic checkpoint hand-off to a private Hugging Face Hub repo. Without an HF_TOKEN Kaggle secret this cell is inert; the manual download/re-attach resume path stays unchanged.
with stage("SYNC-AUX", "Define optional automatic checkpoint sync primitives"):
    import threading

    def _detect_hf_token() -> str | None:
        token = os.environ.get("HF_TOKEN") or os.environ.get("HF_HUB_TOKEN")
        if token and token.strip():
            token = token.strip()
        else:
            token = None
            try:
                from kaggle_secrets import UserSecretsClient
                _client = UserSecretsClient()
                for secret_name in ("HF_TOKEN", "HF_HUB_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_TOKEN", "THRIVELENS_HF_TOKEN", "ThriveLense"):
                    try:
                        candidate = str(_client.get_secret(secret_name))
                    except Exception:
                        continue
                    if candidate:
                        token = candidate.strip()
                        break
            except Exception as exc:
                print(f"[SYNC] Kaggle secrets are unavailable ({type(exc).__name__}: {exc}); continuing without an HF token.")
        if token:
            os.environ["HF_TOKEN"] = token
            print("[SYNC] HF_TOKEN secret detected; automatic remote resume and private adapter sync are ON.")
        return token

    def _hub_api():
        if not AUTO_SYNC:
            return None
        token = _detect_hf_token()
        if not token:
            return None
        try:
            from huggingface_hub import HfApi
            return HfApi(token=token)
        except Exception as exc:
            print(f"[SYNC] huggingface_hub unavailable ({type(exc).__name__}): {exc}; auto-sync stays off.")
            return None

    def _resume_repo_id(api) -> str:
        try:
            from huggingface_hub import whoami
            username = whoami(token=api.token or _detect_hf_token())["name"]
        except Exception:
            username = "ntatemasoabi"
        return f"{username}/{RESUME_REPO_SHORT}"

    def _remote_step(name: str) -> int:
        try:
            return int(name.rsplit("-", 1)[1])
        except Exception:
            return -1

    def _checkpoint_synced_complete(path: Path) -> bool:
        state_file = path / "trainer_state.json"
        if not state_file.exists():
            return False
        try:
            state = json.loads(state_file.read_text(encoding="utf-8"))
            if int(state.get("global_step") or 0) <= 0:
                return False
        except Exception:
            return False
        files = list(path.iterdir())
        if not any((item.name == "model_states.pt") or ("optim" in item.name) or (item.name == "pytorch_model.bin") or (item.name == "adapter_model.safetensors") for item in files):
            return False
        try:
            latest = max(item.stat().st_mtime for item in files)
        except Exception:
            return False
        return (time.time() - latest) >= SYNC_SETTLE_SECONDS

    def _evict_old_remote(api, repo: str, token: str | None) -> None:
        prefix = "resume/"
        try:
            names = sorted(
                path.removeprefix(prefix)
                for path in api.list_repo_files(repo_id=repo, repo_type="model", token=token)
                if path.startswith(prefix)
            )
        except Exception as exc:
            print(f"[SYNC] Remote listing failed ({type(exc).__name__}): {exc}; no eviction performed.")
            return
        entries = {}
        for name in names:
            entries.setdefault(name.split("/", 1)[0], []).append(name)
        obsolete = sorted(entries, key=_remote_step)[max(0, len(entries) - SYNC_REMOTE_HISTORY):]
        for top in obsolete:
            for name in entries[top]:
                try:
                    api.delete_file(repo_id=repo, path_in_repo=prefix + name, repo_type="model", commit_message=f"Drop obsolete {top}", token=token)
                except Exception as exc:
                    print(f"[SYNC] Cleanup of {name} failed ({type(exc).__name__}): {exc}")

    def push_resume_checkpoint(local_dir: Path) -> bool:
        api = _hub_api()
        if api is None:
            return False
        token = _detect_hf_token()
        repo = _resume_repo_id(api)
        name = local_dir.name
        try:
            api.create_repo(repo_id=repo, repo_type="model", private=True, exist_ok=True, token=token)
            _overrides_file = OUTPUT_DIR / "budget_fit_override.json"
            if _overrides_file.exists():
                try:
                    (local_dir / _overrides_file.name).write_text(_overrides_file.read_text(encoding="utf-8"), encoding="utf-8")
                except Exception:
                    pass
            api.upload_folder(
                folder_path=str(local_dir),
                repo_id=repo,
                repo_type="model",
                path_in_repo=f"resume/{name}",
                commit_message=f"ThriveLens resume checkpoint {name}",
                token=token,
            )
            _evict_old_remote(api, repo, token)
            return True
        except Exception as exc:
            print(f"[SYNC] Push of {name} failed ({type(exc).__name__}): {exc}; training continues uninterrupted.")
            return False

    def push_final_adapter() -> bool:
        if not ((OUTPUT_DIR / "adapter_config.json").exists() and (OUTPUT_DIR / "adapter_model.safetensors").exists()):
            return False
        state_file = OUTPUT_DIR / "trainer_state.json"
        if not state_file.exists():
            return False
        try:
            completed_epoch = float(json.loads(state_file.read_text(encoding="utf-8")).get("epoch") or 0)
        except Exception:
            return False
        if completed_epoch < float(EPOCHS) - 1e-6:
            return False
        api = _hub_api()
        if api is None:
            return False
        token = _detect_hf_token()
        repo = _resume_repo_id(api)
        try:
            api.create_repo(repo_id=repo, repo_type="model", private=True, exist_ok=True, token=token)
            api.upload_folder(
                folder_path=str(OUTPUT_DIR),
                repo_id=repo,
                repo_type="model",
                path_in_repo="resume/FINAL",
                commit_message="ThriveLens completed adapter",
                token=token,
                ignore_patterns=["checkpoint-*", "*.log", "remote_resume_stage"],
            )
            return True
        except Exception as exc:
            print(f"[SYNC] Final adapter push failed ({type(exc).__name__}): {exc}")
            return False

    def _remote_resume_repo_exists(api, repo: str, token: str | None) -> bool:
        try:
            api.model_info(repo_id=repo, repo_type="model", token=token)
            return True
        except Exception:
            return False

    def restore_remote_resume() -> int:
        if list(OUTPUT_DIR.glob("checkpoint-*")):
            print("[SYNC] Local checkpoints already exist; remote restore skipped.")
            return 0
        api = _hub_api()
        if api is None:
            print("[SYNC] No HF_TOKEN secret detected; automatic remote resume is OFF. The manual download/re-attach path remains available.")
            return 0
        token = _detect_hf_token()
        repo = _resume_repo_id(api)
        if not _remote_resume_repo_exists(api, repo, token):
            _fallback_repo = f"ntatemasoabi/{RESUME_REPO_SHORT}"
            if _fallback_repo != repo and _remote_resume_repo_exists(api, _fallback_repo, token):
                print(f"[SYNC] Resume checkpoint repo lives at {_fallback_repo}; restoring from there.")
                repo = _fallback_repo
            else:
                print(f"[SYNC] No remote resume repo yet ({repo}); starting from scratch.")
                return 0
        try:
            from huggingface_hub import snapshot_download
            stage_dir = WORK / "remote_resume_stage"
            shutil.rmtree(stage_dir, ignore_errors=True)
            stage_dir.mkdir(parents=True, exist_ok=True)
            snapshot_download(repo_id=repo, repo_type="model", token=token, local_dir=stage_dir, allow_patterns="resume/*", ignore_patterns=[".gitattributes"])
        except Exception as exc:
            print(f"[SYNC] Remote restore failed ({type(exc).__name__}): {exc}; continuing with local/manual resume.")
            return 0
        restored = 0
        resume_root = stage_dir / "resume"
        for source in sorted(resume_root.glob("checkpoint-*"), key=lambda p: _remote_step(p.name), reverse=True):
            if not _checkpoint_synced_complete(source):
                print(f"[SYNC] Skipping incomplete remote checkpoint {source.name}.")
                continue
            shutil.copytree(source, OUTPUT_DIR / source.name, dirs_exist_ok=True)
            restored += 1
        final_src = resume_root / "FINAL"
        if final_src.is_dir():
            for item in final_src.iterdir():
                try:
                    shutil.copy2(item, OUTPUT_DIR / item.name)
                except Exception as exc:
                    print(f"[SYNC] Final-adapter restore skipped {item.name} ({type(exc).__name__}): {exc}")
            print("[SYNC] Completed-adapter markers restored; STAGE 16 will treat the run as finished.")
        print(f"[SYNC] Restored {restored} remote checkpoint(s) into {OUTPUT_DIR} for automatic resume.")
        return restored

    class ResumeSyncWatcher:
        def __init__(self) -> None:
            self._stop = threading.Event()
            self._thread = None
            self._seen = set()

        def start(self) -> None:
            self._thread = threading.Thread(target=self._run, name="thrivelens-resume-sync", daemon=True)
            self._thread.start()
            print("[SYNC] Checkpoint watcher started; every completed checkpoint is mirrored automatically.")

        def stop(self) -> None:
            self._stop.set()
            if self._thread and self._thread.is_alive():
                self._thread.join(timeout=max(SYNC_POLL_SECONDS * 2, 120))

        def final_sweep(self) -> None:
            for path in sorted(OUTPUT_DIR.glob("checkpoint-*")):
                if path.name in self._seen:
                    continue
                if _checkpoint_synced_complete(path):
                    if push_resume_checkpoint(path):
                        self._seen.add(path.name)
            push_final_adapter()

        def _run(self) -> None:
            while not self._stop.wait(SYNC_POLL_SECONDS):
                try:
                    self.final_sweep()
                except Exception as exc:
                    print(f"[SYNC] Watcher loop error ({type(exc).__name__}): {exc}")




[STAGE SYNC-AUX] Define optional automatic checkpoint sync primitives
[START] 2026-09-08T19:30:46.873596+00:00
[STAGE SYNC-AUX OK] elapsed=0.00s


In [13]:
# pyright: ignore
# 10 — Stage, patch, compile, and fingerprint the commit-pinned Qwen3-VL trainer.
with stage("10", "Activate robust raw-ZIP/extracted-dataset media loading and model instrumentation"):
    TRAINER_ROOT = WORK / "training" / "qwen-vl-finetune"
    DATA_UTILS = TRAINER_ROOT / "src" / "dataset" / "data_utils.py"
    SFT_DATASET = TRAINER_ROOT / "src" / "dataset" / "sft_dataset.py"
    TRAIN_SFT = TRAINER_ROOT / "src" / "train" / "train_sft.py"
    ZERO2_CONFIG = TRAINER_ROOT / "scripts" / "zero2.json"
    _pinned_trainer_missing = [
        str(required.relative_to(TRAINER_ROOT))
        for required in (DATA_UTILS, SFT_DATASET, TRAIN_SFT, ZERO2_CONFIG)
        if not required.exists()
    ]
    if _pinned_trainer_missing:
        print(f"[TRAINER] Staged trainer tree is incomplete; missing {_pinned_trainer_missing}. Attempting self-heal.", flush=True)

    expected_commit = (TRAINER_ROOT / "PINNED_COMMIT.txt").read_text(encoding="utf-8").strip()
    assert expected_commit == "70c7b2fcb0e276b1fa4b136852e9b862ce8730fa"
    print(f"[TRAINER] pinned commit={expected_commit}")
    deepspeed_config = json.loads(ZERO2_CONFIG.read_text(encoding="utf-8"))
    assert deepspeed_config.get("zero_optimization", {}).get("stage") == 2
    assert deepspeed_config.get("train_micro_batch_size_per_gpu") == "auto"
    assert deepspeed_config.get("train_batch_size") == "auto"
    assert deepspeed_config.get("gradient_accumulation_steps") == "auto"
    assert deepspeed_config.get("fp16", {}).get("enabled") == "auto"
    assert deepspeed_config.get("bf16", {}).get("enabled") == "auto"
    assert "optimizer" not in deepspeed_config and "scheduler" not in deepspeed_config, (
        "DeepSpeed must not override the Cell 02 optimizer or scheduler."
    )
    print("[DEEPSPEED CONTRACT] ZeRO-2 uses auto batch/precision fields and does not override optimizer or scheduler.")

    _trainer_usable = all(required.exists() for required in (DATA_UTILS, SFT_DATASET, TRAIN_SFT, ZERO2_CONFIG))
    if _trainer_usable:
        for _source in (DATA_UTILS, SFT_DATASET, TRAIN_SFT):
            try:
                compile(_source.read_text(encoding="utf-8"), str(_source), "exec")
            except (SyntaxError, OSError):
                _trainer_usable = False
                break
    if not _trainer_usable:
        _restored_from = None
        _archive = os.environ.get("THRIVELENS_ARCHIVE")
        _root = os.environ.get("THRIVELENS_DATASET_ROOT")

        def _pristine_trainer_tree():
            _search_bases = []
            if _root and Path(_root).is_dir():
                _search_bases.append(Path(_root))
            for _candidate in sorted(Path("/kaggle/input").rglob("qwen-vl-finetune")):
                if not str(_candidate).startswith(str(WORK)):
                    _search_bases.append(_candidate)
                    break
            for _base in _search_bases:
                for _hit in sorted(_base.rglob("train_sft.py")):
                    for _parent in _hit.parents:
                        if (_parent / "scripts" / "zero2.json").exists() and (_parent / "src" / "dataset" / "data_utils.py").exists():
                            return _parent
            return None

        _pristine = _pristine_trainer_tree()
        if _pristine is not None:
            TRAINER_ROOT.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(_pristine, TRAINER_ROOT, dirs_exist_ok=True)
            _restored_from = str(_pristine)
        elif _archive and os.path.isfile(_archive):
            with zipfile.ZipFile(_archive) as _zf:
                _trainer_prefix = next(
                    (name.split("/src/train/train_sft.py", 1)[0] for name in _zf.namelist()
                     if name.endswith("/src/train/train_sft.py")),
                    None,
                )
                if _trainer_prefix:
                    for _name in _zf.namelist():
                        if _name.startswith(_trainer_prefix + "/"):
                            _zf.extract(_name, WORK)
                    TRAINER_ROOT.parent.mkdir(parents=True, exist_ok=True)
                    _extracted = WORK / _trainer_prefix
                    if _extracted.is_dir() and _extracted != TRAINER_ROOT:
                        shutil.copytree(_extracted, TRAINER_ROOT, dirs_exist_ok=True)
                    _restored_from = f"archive {_archive}"
        if _restored_from:
            print(f"[TRAINER SELF-HEAL] restored pristine trainer files from {_restored_from} (files were missing or un-parseable).", flush=True)
        elif all(required.exists() for required in (DATA_UTILS, SFT_DATASET, TRAIN_SFT, ZERO2_CONFIG)):
            _train_text = TRAIN_SFT.read_text(encoding="utf-8")
            _lines = _train_text.split("\n")
            _repaired = False
            for _i, _l in enumerate(_lines):
                if _l.strip().startswith('if "visual" not in training_args.lora_namespan_exclude:'):
                    _if_indent = len(_l) - len(_l.lstrip(" "))
                    for _j in range(_i + 1, len(_lines)):
                        _sj = _lines[_j].strip()
                        if _sj.startswith("training_args.lora_namespan_exclude.append("):
                            _ap_indent = len(_lines[_j]) - len(_lines[_j].lstrip(" "))
                            if _ap_indent <= _if_indent:
                                _lines[_j] = " " * (_if_indent + 4) + _sj
                                _repaired = True
                            break
                    break
            if _repaired:
                TRAIN_SFT.write_text("\n".join(_lines), encoding="utf-8")
                print("[TRAINER SELF-HEAL] repaired corrupt visual-exclusion indentation in place (no pristine source found).", flush=True)
            else:
                raise RuntimeError("Trainer files are un-parseable and no pristine source could be located for restore.")
        else:
            raise RuntimeError(
                "Pinned trainer files are missing and no pristine source (dataset root, /kaggle/input, or archive) "
                "could be located for restore. Attach the ThriveLens dataset and rerun."
            )
    assert all(required.exists() for required in (DATA_UTILS, SFT_DATASET, TRAIN_SFT, ZERO2_CONFIG)), (
        "Pinned trainer files are still missing after self-heal."
    )

    data_text = DATA_UTILS.read_text(encoding="utf-8")
    for import_line in ("import io", "import os", "import zipfile", "from PIL import Image"):
        if import_line not in data_text:
            data_text = data_text.replace("import re\n", f"import re\n{import_line}\n", 1)
    resolver = '''
# THRIVELENS_RUNTIME_MEDIA_RESOLVER_V2
@lru_cache(maxsize=16)
def _thrivelens_zip_handle(archive_path, process_id):
    # process_id is part of the cache key so forked DataLoader workers never share a handle.
    return zipfile.ZipFile(archive_path, "r")

def _resolve_thrivelens_image(image_path):
    if not isinstance(image_path, str) or not image_path.startswith("zip://"):
        return image_path
    member = image_path[len("zip://"):].lstrip("/" + chr(92))
    archive_path = os.environ.get("THRIVELENS_ARCHIVE")
    dataset_root = os.environ.get("THRIVELENS_DATASET_ROOT")
    if archive_path:
        raw = _thrivelens_zip_handle(archive_path, os.getpid()).read(member)
        with Image.open(io.BytesIO(raw)) as image:
            return image.convert("RGB").copy()
    if dataset_root:
        with Image.open(os.path.join(dataset_root, member)) as image:
            return image.convert("RGB").copy()
    raise RuntimeError("Set THRIVELENS_ARCHIVE or THRIVELENS_DATASET_ROOT before loading data")

'''
    if "THRIVELENS_RUNTIME_MEDIA_RESOLVER_V2" not in data_text:
        data_text = data_text.replace("def get_image_info(image_path, min_pixel, max_pixel, width, height, image_patch_size):", resolver + "def get_image_info(image_path, min_pixel, max_pixel, width, height, image_patch_size):", 1)
    if "image_path = _resolve_thrivelens_image(image_path)" not in data_text:
        signature = "def get_image_info(image_path, min_pixel, max_pixel, width, height, image_patch_size):\n"
        data_text = data_text.replace(signature, signature + "    image_path = _resolve_thrivelens_image(image_path)\n", 1)
    data_text = data_text.replace("image_path = _resolve_zip_image(image_path)", "image_path = _resolve_thrivelens_image(image_path)")
    DATA_UTILS.write_text(data_text, encoding="utf-8")

    sft_text = SFT_DATASET.read_text(encoding="utf-8")
    sft_text = sft_text.replace('if not image_file.startswith("http"):', 'if not image_file.startswith(("http", "zip://")):')
    SFT_DATASET.write_text(sft_text, encoding="utf-8")

    train_text = TRAIN_SFT.read_text(encoding="utf-8")
    if "import json" not in train_text:
        train_text = train_text.replace("import os\n", "import os\nimport json\n", 1)
    import re as _re
    _visual_exclusion = _re.compile(r"^(\s*)training_args\.lora_namespan_exclude \+= \[\s*[\"']visual[\"']\s*\][^\n]*$", _re.MULTILINE)
    _visual_matches = list(_visual_exclusion.finditer(train_text))
    if not _visual_matches:
        _visual_candidates = [l.strip() for l in train_text.splitlines() if "lora_namespan_exclude" in l]
        print("[TRAINER INFO] visual-exclusion `+= [\"visual\"]` line not found in train_sft.py; skipping that cosmetic patch (non-fatal).", flush=True)
        print(f"[TRAINER INFO] lines containing lora_namespan_exclude: {_visual_candidates}", flush=True)
    else:
        _visual_indent = _visual_matches[0].group(1) or "        "
        _visual_exclusion_patch = (
            _visual_indent + 'if "visual" not in training_args.lora_namespan_exclude:\n'
            + _visual_indent + '    training_args.lora_namespan_exclude.append("visual")'
        )
        train_text = _visual_exclusion.sub(lambda _m: _visual_exclusion_patch, train_text, count=1)
    checkpointing_assignment = f"training_args.gradient_checkpointing_kwargs = {GRADIENT_CHECKPOINTING_KWARGS!r}"
    checkpointing_found = 0
    if checkpointing_assignment in train_text:
        print(f"[TRAINER INFO] gradient_checkpointing_kwargs already {GRADIENT_CHECKPOINTING_KWARGS!r}; skipping (idempotent).", flush=True)
    else:
        for legacy_assignment in (
            'training_args.gradient_checkpointing_kwargs = {"use_reentrant": False}',
            'training_args.gradient_checkpointing_kwargs = {"use_reentrant": True}',
        ):
            checkpointing_found += train_text.count(legacy_assignment)
            train_text = train_text.replace(legacy_assignment, checkpointing_assignment)
        if checkpointing_found >= 2:
            print(f"[TRAINER PATCH] gradient_checkpointing_kwargs={GRADIENT_CHECKPOINTING_KWARGS}", flush=True)
        else:
            _ckpt_candidates = [l.strip() for l in train_text.splitlines() if "gradient_checkpointing_kwargs" in l]
            print("[TRAINER INFO] gradient_checkpointing_kwargs in unexpected form; leaving unchanged (non-fatal).", flush=True)
            print(f"[TRAINER INFO] lines containing gradient_checkpointing_kwargs: {_ckpt_candidates}", flush=True)
    if "THRIVELENS_DISK_EVICT_V1" not in train_text:
        _evict_anchor = "    if list(pathlib.Path(training_args.output_dir).glob(\"checkpoint-*\")):\n"
        _evict_insert = (
            "# THRIVELENS_DISK_EVICT_V1\n"
            "    if os.environ.get(\"THRIVELENS_FREE_BASE_CACHE_AFTER_LOAD\") == \"1\" and int(os.environ.get(\"LOCAL_RANK\", \"-1\")) == 0:\n"
            "        import shutil as _cache_shutil, threading as _cache_threading, time as _cache_time\n"
            "        def _thrivelens_evict_base_cache():\n"
            "            _cache_time.sleep(240)\n"
            "            _scratch_dir = os.path.dirname(os.environ.get(\"HF_HOME\", \"/kaggle/working\")) or \"/kaggle/working\"\n"
            "            _before = _cache_shutil.disk_usage(_scratch_dir).free\n"
            "            for _cache_root in (os.environ.get(\"HF_HOME\", \"\"), os.environ.get(\"TRANSFORMERS_CACHE\", \"\"), os.environ.get(\"HF_HUB_CACHE\", \"\")):\n"
            "                if _cache_root:\n"
            "                    _cache_shutil.rmtree(_cache_root, ignore_errors=True)\n"
            "            _after = _cache_shutil.disk_usage(_scratch_dir).free\n"
            "            print(f\"[THRIVELENS-CACHE] Evicted the base-model hub cache and freed {(_after - _before) / 2**30:.2f} GiB on /kaggle/working for checkpoint storage.\", flush=True)\n"
            "        _cache_threading.Thread(target=_thrivelens_evict_base_cache, name=\"thrivelens-disk-evict\", daemon=True).start()\n"
            "        print(\"[THRIVELENS-CACHE] Base-model hub cache will be evicted right after the trainer finishes loading; the 20GiB quota is reserved for checkpoints and artifacts.\", flush=True)\n"
        )
        assert _evict_anchor in train_text, "Could not locate trainer.train() dispatch block for the cache-eviction patch"
        train_text = train_text.replace(_evict_anchor, _evict_insert + _evict_anchor, 1)
    instrumentation = '''
        # THRIVELENS_MODEL_SUMMARY_V2
        if local_rank == 0 or local_rank == "0" or local_rank is None:
            os.makedirs(training_args.output_dir, exist_ok=True)
            total_parameters = sum(parameter.numel() for parameter in model.parameters())
            trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
            trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
            visual_trainable = [name for name in trainable_names if "visual" in name or "merger" in name]
            active_peft_config = model.peft_config.get("default")
            peft_report = {
                "rank": active_peft_config.r,
                "alpha": active_peft_config.lora_alpha,
                "dropout": active_peft_config.lora_dropout,
                "bias": active_peft_config.bias,
                "target_module_count": len(active_peft_config.target_modules),
                "target_module_examples": sorted(active_peft_config.target_modules)[:100],
            }
            assert peft_report["rank"] == training_args.lora_rank
            assert peft_report["alpha"] == training_args.lora_alpha
            assert abs(peft_report["dropout"] - training_args.lora_dropout) < 1e-12
            assert peft_report["bias"] == training_args.lora_bias
            assert peft_report["target_module_count"] > 0
            dtype_counts = {}
            for parameter in model.parameters():
                key = str(parameter.dtype)
                dtype_counts[key] = dtype_counts.get(key, 0) + parameter.numel()
            report = {
                "model_class": model.__class__.__name__,
                "total_parameter_elements": total_parameters,
                "trainable_parameter_elements": trainable_parameters,
                "trainable_percent": 100.0 * trainable_parameters / max(total_parameters, 1),
                "trainable_tensor_count": len(trainable_names),
                "LoRA_target_tensor_examples": trainable_names[:100],
                "visual_or_merger_trainable_tensors": visual_trainable,
                "peft_configuration": peft_report,
                "gradient_checkpointing": training_args.gradient_checkpointing,
                "gradient_checkpointing_kwargs": training_args.gradient_checkpointing_kwargs,
                "parameter_elements_by_dtype": dtype_counts,
            }
            print("\n" + "=" * 100, flush=True)
            print("THRIVELENS MODEL + QLORA TRAINABLE-PARAMETER SUMMARY", flush=True)
            model.print_trainable_parameters()
            print(json.dumps(report, indent=2), flush=True)
            assert not visual_trainable, f"Visual parameters unexpectedly trainable under 4-bit QLoRA: {visual_trainable[:10]}"
            with open(os.path.join(training_args.output_dir, "model_parameter_summary.json"), "w", encoding="utf-8") as handle:
                json.dump(report, handle, indent=2)
            print("=" * 100 + "\n", flush=True)
'''
    instrumentation = instrumentation.replace('"' + chr(10) + '"', 'chr(10)')
    marker = "        model = get_peft_model(model, peft_config)\n"
    if "THRIVELENS_MODEL_SUMMARY_V2" not in train_text:
        assert marker in train_text, "Could not locate LoRA insertion point in pinned trainer"
        train_text = train_text.replace(marker, marker + instrumentation, 1)
    TRAIN_SFT.write_text(train_text, encoding="utf-8")

    for source in (DATA_UTILS, SFT_DATASET, TRAIN_SFT):
        subprocess.run([sys.executable, "-m", "py_compile", str(source)], check=True)
        print(f"[TRAINER] compiled {source.relative_to(TRAINER_ROOT)} sha256={sha256_file(source, progress_gib=100):s}")
    os.environ["PYTHONPATH"] = str(TRAINER_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")




[STAGE 10] Activate robust raw-ZIP/extracted-dataset media loading and model instrumentation
[START] 2026-09-08T19:30:46.904533+00:00
[TRAINER] pinned commit=70c7b2fcb0e276b1fa4b136852e9b862ce8730fa
[DEEPSPEED CONTRACT] ZeRO-2 uses auto batch/precision fields and does not override optimizer or scheduler.
[TRAINER PATCH] gradient_checkpointing_kwargs={'use_reentrant': False}
[TRAINER] compiled src/dataset/data_utils.py sha256=8fbacc5a8ba000aaaeb23084644bf490c3d97a3ae68e6de1d669850be227fab8
[TRAINER] compiled src/dataset/sft_dataset.py sha256=31e0ed6bec3c9479be8fdef10f410604b17396794a8a4f736a6725c62a5b3794
[TRAINER] compiled src/train/train_sft.py sha256=f46b0de30f2157a584cc17d5cf998f84c2197f1dc48c436ac354afe58e812a82
[STAGE 10 OK] elapsed=0.29s


In [14]:
# pyright: ignore
# 11 — Food-R1 architecture, processor, tokenizer, and language-length report before weight loading.
with stage("11", "Inspect the pretrained Food-R1 architecture and input contract"):
    import numpy as np
    from transformers import AutoConfig, AutoProcessor

    model_config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        min_pixels=MIN_IMAGE_PIXELS,
        max_pixels=MAX_IMAGE_PIXELS,
    )
    text_config = getattr(model_config, "text_config", model_config)
    vision_config = getattr(model_config, "vision_config", None)
    architecture_report = {
        "model_id": MODEL_ID,
        "model_type": getattr(model_config, "model_type", None),
        "architectures": getattr(model_config, "architectures", None),
        "processor_class": processor.__class__.__name__,
        "tokenizer_class": processor.tokenizer.__class__.__name__,
        "vocabulary_size": len(processor.tokenizer),
        "pad_token": processor.tokenizer.pad_token,
        "pad_token_id": processor.tokenizer.pad_token_id,
        "eos_token": processor.tokenizer.eos_token,
        "text_hidden_size": getattr(text_config, "hidden_size", None),
        "text_layers": getattr(text_config, "num_hidden_layers", None),
        "attention_heads": getattr(text_config, "num_attention_heads", None),
        "key_value_heads": getattr(text_config, "num_key_value_heads", None),
        "intermediate_size": getattr(text_config, "intermediate_size", None),
        "maximum_positions": getattr(text_config, "max_position_embeddings", None),
        "vision_model_type": getattr(vision_config, "model_type", None) if vision_config else None,
        "vision_depth": getattr(vision_config, "depth", None) if vision_config else None,
        "vision_hidden_size": getattr(vision_config, "hidden_size", None) if vision_config else None,
        "training_image_pixel_range": [MIN_IMAGE_PIXELS, MAX_IMAGE_PIXELS],
    }
    print_json(architecture_report)

    del processor



[STAGE 11] Inspect the pretrained Food-R1 architecture and input contract
[START] 2026-09-08T19:30:47.209464+00:00


config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (2 outputs left out)

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

{
  "model_id": "zy12123/Food-R1",
  "model_type": "qwen3_vl",
  "architectures": [
    "Qwen3VLForConditionalGeneration"
  ],
  "processor_class": "Qwen3VLProcessor",
  "tokenizer_class": "Qwen2Tokenizer",
  "vocabulary_size": 151669,
  "pad_token": "<|endoftext|>",
  "pad_token_id": 151643,
  "eos_token": "<|im_end|>",
  "text_hidden_size": 4096,
  "text_layers": 36,
  "attention_heads": 32,
  "key_value_heads": 8,
  "intermediate_size": 12288,
  "maximum_positions": 262144,
  "vision_model_type": "qwen3_vl",
  "vision_depth": 27,
  "vision_hidden_size": 1152,
  "training_image_pixel_range": [
    65536,
    262144
  ]
}
{
  "sample_count": 1000,
  "language_tokens_only": {
    "minimum": 31,
    "median": 44.0,
    "p95": 812.2499999999998,
    "p99": 1341.5199999999995,
    "maximum": 2197
  },
  "by_task_maximum": {
    "food_classification": 35,
    "produce_variety_classification": 45,
    "nutrition_estimation": 97,
    "full_food_analysis": 2197,
    "ingredient_recognition": 

In [15]:
# pyright: ignore
# 12 — A real NCCL all-reduce proves that both GPUs can communicate before model allocation.
with stage("12", "Run a two-process CUDA/NCCL communication test"):
    ddp_script = WORK / "dual_gpu_nccl_preflight.py"
    ddp_script.write_text('''from __future__ import annotations
import json, os, socket
import torch
import torch.distributed as dist

dist.init_process_group("nccl")
rank = dist.get_rank()
world = dist.get_world_size()
torch.cuda.set_device(rank)
device = torch.device(f"cuda:{rank}")
value = torch.tensor([float(rank + 1)], device=device)
dist.all_reduce(value, op=dist.ReduceOp.SUM)
expected_world = int(os.environ["THRIVELENS_EXPECTED_WORLD_SIZE"])
expected_sum = expected_world * (expected_world + 1) / 2
assert world == expected_world, (world, expected_world)
assert value.item() == expected_sum, (value.item(), expected_sum)
payload = {
    "rank": rank, "world_size": world, "pid": os.getpid(), "host": socket.gethostname(),
    "device": str(device), "gpu": torch.cuda.get_device_name(rank),
    "all_reduce_expected": expected_sum, "all_reduce_received": value.item(),
}
print("[NCCL PREFLIGHT] " + json.dumps(payload), flush=True)
dist.barrier()
dist.destroy_process_group()
''', encoding="utf-8")
    subprocess.run([sys.executable, "-m", "py_compile", str(ddp_script)], check=True)
    ddp_env = {
        **os.environ,
        "CUDA_VISIBLE_DEVICES": ",".join(str(index) for index in range(WORLD_SIZE)),
        "NCCL_DEBUG": "WARN",
        "NCCL_P2P_DISABLE": "1",
        "THRIVELENS_EXPECTED_WORLD_SIZE": str(WORLD_SIZE),
    }
    run_streaming(
        ["torchrun", "--standalone", f"--nproc_per_node={WORLD_SIZE}", str(ddp_script)],
        env=ddp_env,
        log_path=WORK / "dual_gpu_nccl_preflight.log",
        label="NCCL-PREFLIGHT",
    )
    preflight_log = (WORK / "dual_gpu_nccl_preflight.log").read_text(encoding="utf-8")   
    assert preflight_log.count("[NCCL PREFLIGHT]") >= WORLD_SIZE, "Every rank must report a successful all-reduce"
    print(f"[NCCL PREFLIGHT] Exactly {WORLD_SIZE} ranks completed the expected sum on separate CUDA devices.")


[STAGE 12] Run a two-process CUDA/NCCL communication test
[START] 2026-09-08T19:30:59.657883+00:00
[NCCL-PREFLIGHT] cwd=/kaggle/working
[NCCL-PREFLIGHT] command:
torchrun --standalone --nproc_per_node=2 /kaggle/working/thrivelens_setup/dual_gpu_nccl_preflight.py
W0908 19:31:01.588000 406 torch/distributed/run.py:852] 
W0908 19:31:01.588000 406 torch/distributed/run.py:852] *****************************************
W0908 19:31:01.588000 406 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0908 19:31:01.588000 406 torch/distributed/run.py:852] *****************************************
[W908 19:31:01.988496484 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W908 19:31:03.736645202 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=

In [16]:
# pyright: ignore
# 13 — Construct auditable DeepSpeed commands; no shell interpolation and no early-stopping callback.
with stage("13", "Build the smoke-test and full-training launch specifications"):
    import shlex

    deepspeed_executable = shutil.which("deepspeed")
    assert deepspeed_executable, "DeepSpeed executable was not installed"
    cli_bool = lambda value: str(bool(value))
    assert LORA_TARGET_MODULES == "all-linear", "This trainer maps all-linear to --num_lora_modules -1"
    num_lora_modules = -1
    expected_compute_dtype = "float16" if FP16 else ("bfloat16" if BF16 else "float32")
    assert BNB_4BIT_COMPUTE_DTYPE == expected_compute_dtype

    def build_training_command(data_path: Path, eval_path: Path | None, output_dir: Path, *, smoke: bool):
        command = [
            deepspeed_executable, "--num_gpus", str(WORLD_SIZE), str(TRAIN_SFT),
            "--deepspeed", str(ZERO2_CONFIG),
            "--model_id", MODEL_ID,
            "--data_path", str(data_path),
            "--image_folder", "/unused/thrivelens_virtual_media",
            "--remove_unused_columns", cli_bool(REMOVE_UNUSED_COLUMNS),
            "--bits", "4" if LOAD_IN_4BIT else "16",
            "--double_quant", cli_bool(BNB_4BIT_USE_DOUBLE_QUANT),
            "--quant_type", BNB_4BIT_QUANT_TYPE,
            "--lora_enable", cli_bool(LORA_ENABLE),
            "--vision_lora", cli_bool(VISION_LORA),
            "--use_dora", cli_bool(USE_DORA),
            "--lora_namespan_exclude", repr(list(LORA_EXCLUDED_NAME_SPANS)),
            "--lora_rank", str(LORA_RANK),
            "--lora_alpha", str(LORA_ALPHA),
            "--lora_dropout", str(LORA_DROPOUT),
            "--lora_bias", LORA_BIAS,
            "--num_lora_modules", str(num_lora_modules),
            "--freeze_llm", cli_bool(FREEZE_LLM),
            "--freeze_vision_tower", cli_bool(FREEZE_VISION_TOWER),
            "--freeze_merger", cli_bool(FREEZE_MERGER),
            "--unfreeze_topk_llm", str(UNFREEZE_TOPK_LLM),
            "--unfreeze_topk_vision", str(UNFREEZE_TOPK_VISION),
            "--fp16", cli_bool(FP16),
            "--bf16", cli_bool(BF16),
            "--tf32", cli_bool(TF32),
            "--disable_flash_attn2", cli_bool(DISABLE_FLASH_ATTENTION_2),
            "--use_liger_kernel", cli_bool(USE_LIGER_KERNEL),
            "--output_dir", str(output_dir),
            "--per_device_train_batch_size", str(MICRO_BATCH),
            "--per_device_eval_batch_size", str(PER_DEVICE_EVAL_BATCH_SIZE),
            "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
            "--gradient_checkpointing", cli_bool(GRADIENT_CHECKPOINTING),
            "--max_seq_length", str(MAX_SEQUENCE_LENGTH),
            "--image_min_pixels", str(MIN_IMAGE_PIXELS),
            "--image_max_pixels", str(MAX_IMAGE_PIXELS),
            "--learning_rate", str(LEARNING_RATE),
            "--optim", OPTIMIZER,
            "--adam_beta1", str(ADAM_BETA1),
            "--adam_beta2", str(ADAM_BETA2),
            "--adam_epsilon", str(ADAM_EPSILON),
            "--weight_decay", str(WEIGHT_DECAY),
            "--max_grad_norm", str(MAX_GRAD_NORM),
            "--warmup_ratio", str(WARMUP_RATIO),
            "--lr_scheduler_type", LR_SCHEDULER_TYPE,
            "--logging_strategy", LOGGING_STRATEGY,
            "--logging_steps", "1" if smoke else str(LOGGING_STEPS),
            "--logging_first_step", cli_bool(LOG_FIRST_STEP),
            "--logging_nan_inf_filter", cli_bool(LOGGING_NAN_INF_FILTER),
            "--log_level", "info",
            "--log_level_replica", "warning",
            "--disable_tqdm", cli_bool(DISABLE_TQDM),
            "--dataloader_num_workers", str(DATALOADER_WORKERS),
            "--dataloader_prefetch_factor", str(DATALOADER_PREFETCH_FACTOR),
            "--dataloader_pin_memory", cli_bool(DATALOADER_PIN_MEMORY),
            "--dataloader_persistent_workers", cli_bool(DATALOADER_PERSISTENT_WORKERS),
            "--seed", str(SEED),
            "--data_seed", str(SEED),
            "--full_determinism", cli_bool(FULL_DETERMINISM),
            "--ddp_find_unused_parameters", cli_bool(DDP_FIND_UNUSED_PARAMETERS),
            "--ddp_timeout", str(DDP_TIMEOUT_SECONDS),
            "--load_best_model_at_end", cli_bool(LOAD_BEST_MODEL_AT_END),
        ]
        if eval_path is not None:
            command.extend(["--eval_path", str(eval_path)])
        if smoke:
            command.extend([
                "--max_steps", str(SMOKE_STEPS),
                "--num_train_epochs", "1",
                "--eval_strategy", "no",
                "--save_strategy", "no",
                "--report_to", "none",
                "--run_name", "thrivelens-dual-t4-smoke-test",
            ])
        else:
            command.extend([
                "--num_train_epochs", str(EPOCHS),
                "--eval_strategy", EVAL_STRATEGY,
                "--prediction_loss_only", cli_bool(PREDICTION_LOSS_ONLY),
                "--save_strategy", SAVE_STRATEGY,
                "--save_steps", str(SAVE_STEPS),
                "--save_total_limit", str(SAVE_TOTAL_LIMIT),
                "--logging_dir", str(OUTPUT_DIR / "tensorboard"),
                "--report_to", "tensorboard",
                "--run_name", "thrivelens-foodr1-global-300k",
            ])
        return command

    training_env = {
        **os.environ,
        "PYTHONPATH": str(TRAINER_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""),
        "CUDA_VISIBLE_DEVICES": ",".join(str(index) for index in range(WORLD_SIZE)),
        "NCCL_DEBUG": "WARN",
        "NCCL_P2P_DISABLE": "1",
        "TOKENIZERS_PARALLELISM": "false",
        "WANDB_DISABLED": "true",
    }
    FULL_COMMAND = build_training_command(SFT_DIR / "train.json", SFT_DIR / "val.json", OUTPUT_DIR, smoke=False)
    def command_value(command: list[str], option: str) -> str:
        assert command.count(option) == 1, f"Expected exactly one {option}; command contains {command.count(option)}"
        return command[command.index(option) + 1]

    expected_command_contract = {
        "--num_gpus": WORLD_SIZE, "--remove_unused_columns": cli_bool(REMOVE_UNUSED_COLUMNS),
        "--bits": 4 if LOAD_IN_4BIT else 16,
        "--double_quant": cli_bool(BNB_4BIT_USE_DOUBLE_QUANT), "--quant_type": BNB_4BIT_QUANT_TYPE,
        "--lora_enable": cli_bool(LORA_ENABLE), "--vision_lora": cli_bool(VISION_LORA), "--use_dora": cli_bool(USE_DORA),
        "--lora_rank": LORA_RANK, "--lora_alpha": LORA_ALPHA, "--lora_dropout": LORA_DROPOUT,
        "--lora_namespan_exclude": repr(list(LORA_EXCLUDED_NAME_SPANS)),
        "--lora_bias": LORA_BIAS, "--num_lora_modules": num_lora_modules,
        "--freeze_llm": cli_bool(FREEZE_LLM), "--freeze_vision_tower": cli_bool(FREEZE_VISION_TOWER),
        "--freeze_merger": cli_bool(FREEZE_MERGER), "--unfreeze_topk_llm": UNFREEZE_TOPK_LLM,
        "--unfreeze_topk_vision": UNFREEZE_TOPK_VISION,
        "--fp16": cli_bool(FP16), "--bf16": cli_bool(BF16), "--tf32": cli_bool(TF32),
        "--disable_flash_attn2": cli_bool(DISABLE_FLASH_ATTENTION_2), "--use_liger_kernel": cli_bool(USE_LIGER_KERNEL),
        "--per_device_train_batch_size": MICRO_BATCH, "--per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "--gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "--gradient_checkpointing": cli_bool(GRADIENT_CHECKPOINTING), "--max_seq_length": MAX_SEQUENCE_LENGTH,
        "--image_min_pixels": MIN_IMAGE_PIXELS, "--image_max_pixels": MAX_IMAGE_PIXELS,
        "--learning_rate": LEARNING_RATE, "--optim": OPTIMIZER, "--adam_beta1": ADAM_BETA1,
        "--adam_beta2": ADAM_BETA2, "--adam_epsilon": ADAM_EPSILON, "--weight_decay": WEIGHT_DECAY,
        "--max_grad_norm": MAX_GRAD_NORM, "--warmup_ratio": WARMUP_RATIO, "--lr_scheduler_type": LR_SCHEDULER_TYPE,
        "--num_train_epochs": EPOCHS, "--eval_strategy": EVAL_STRATEGY, "--save_strategy": SAVE_STRATEGY,
        "--save_steps": SAVE_STEPS, "--save_total_limit": SAVE_TOTAL_LIMIT,
        "--logging_strategy": LOGGING_STRATEGY, "--logging_steps": LOGGING_STEPS, "--logging_first_step": cli_bool(LOG_FIRST_STEP),
        "--logging_nan_inf_filter": cli_bool(LOGGING_NAN_INF_FILTER), "--disable_tqdm": cli_bool(DISABLE_TQDM),
        "--dataloader_num_workers": DATALOADER_WORKERS, "--dataloader_prefetch_factor": DATALOADER_PREFETCH_FACTOR,
        "--dataloader_pin_memory": cli_bool(DATALOADER_PIN_MEMORY),
        "--dataloader_persistent_workers": cli_bool(DATALOADER_PERSISTENT_WORKERS),
        "--ddp_find_unused_parameters": cli_bool(DDP_FIND_UNUSED_PARAMETERS), "--ddp_timeout": DDP_TIMEOUT_SECONDS,
        "--full_determinism": cli_bool(FULL_DETERMINISM),
        "--load_best_model_at_end": cli_bool(LOAD_BEST_MODEL_AT_END), "--prediction_loss_only": cli_bool(PREDICTION_LOSS_ONLY),
    }
    command_contract_report = {}
    for option, expected in expected_command_contract.items():
        actual = command_value(FULL_COMMAND, option)
        assert actual == str(expected), f"Configuration drift for {option}: expected {expected!r}, got {actual!r}"
        command_contract_report[option] = actual
    assert EARLY_STOPPING is False and "early_stopping" not in " ".join(FULL_COMMAND).lower()
    assert command_value(FULL_COMMAND, "--load_best_model_at_end") == cli_bool(LOAD_BEST_MODEL_AT_END)
    print("[CONFIG PROPAGATION] Every critical Cell 02 setting is present in the executable training command.")
    print_json(command_contract_report)
    (OUTPUT_DIR / "verified_training_argument_contract.json").write_text(json.dumps(command_contract_report, indent=2), encoding="utf-8")
    (OUTPUT_DIR / "full_training_command.json").write_text(json.dumps(FULL_COMMAND, indent=2), encoding="utf-8")
    (OUTPUT_DIR / "full_training_command.sh").write_text("#!/usr/bin/env bash\nset -euo pipefail\n" + shlex.join(FULL_COMMAND) + "\n", encoding="utf-8")
    print("[FULL TRAINING COMMAND]")
    print(shlex.join(FULL_COMMAND))
    print(f"[POLICY] fixed {EPOCHS} epochs; load_best_model_at_end=False; no early-stopping callback")


[STAGE 13] Build the smoke-test and full-training launch specifications
[START] 2026-09-08T19:31:05.860457+00:00
[CONFIG PROPAGATION] Every critical Cell 02 setting is present in the executable training command.
{
  "--num_gpus": "2",
  "--remove_unused_columns": "False",
  "--bits": "4",
  "--double_quant": "True",
  "--quant_type": "nf4",
  "--lora_enable": "True",
  "--vision_lora": "False",
  "--use_dora": "False",
  "--lora_rank": "64",
  "--lora_alpha": "16",
  "--lora_dropout": "0.05",
  "--lora_namespan_exclude": "['lm_head', 'embed_tokens', 'visual']",
  "--lora_bias": "none",
  "--num_lora_modules": "-1",
  "--freeze_llm": "True",
  "--freeze_vision_tower": "True",
  "--freeze_merger": "True",
  "--unfreeze_topk_llm": "0",
  "--unfreeze_topk_vision": "0",
  "--fp16": "True",
  "--bf16": "False",
  "--tf32": "False",
  "--disable_flash_attn2": "True",
  "--use_liger_kernel": "False",
  "--per_device_train_batch_size": "1",
  "--per_device_eval_batch_size": "1",
  "--gradient_

In [17]:
# pyright: ignore
# 14 — Build small, task-stratified train/validation manifests for an end-to-end two-GPU smoke run.
with stage("14", "Create task-stratified smoke-test manifests"):
    task_names = set(summary["sft_tasks"]["train"])
    smoke_train_by_task = defaultdict(list)
    for _, row in iter_jsonl(SFT_DIR / "train.jsonl"):
        task = row["task"]
        if len(smoke_train_by_task[task]) < SMOKE_EXAMPLES_PER_TASK:
            smoke_train_by_task[task].append(row)
        if set(smoke_train_by_task) == task_names and all(len(rows) >= SMOKE_EXAMPLES_PER_TASK for rows in smoke_train_by_task.values()):
            break
    assert set(smoke_train_by_task) == task_names
    smoke_train = [row for task in sorted(smoke_train_by_task) for row in smoke_train_by_task[task]]

    smoke_val_by_task = defaultdict(list)
    val_task_names = set(summary["sft_tasks"]["val"])
    for _, row in iter_jsonl(SFT_DIR / "val.jsonl"):
        task = row["task"]
        if len(smoke_val_by_task[task]) < 2:
            smoke_val_by_task[task].append(row)
        if set(smoke_val_by_task) == val_task_names and all(len(rows) >= 2 for rows in smoke_val_by_task.values()):
            break
    smoke_val = [row for task in sorted(smoke_val_by_task) for row in smoke_val_by_task[task]]

    SMOKE_TRAIN_JSON = WORK / "smoke_train.json"
    SMOKE_VAL_JSON = WORK / "smoke_val.json"
    SMOKE_TRAIN_JSON.write_text(json.dumps(smoke_train, ensure_ascii=False), encoding="utf-8")
    SMOKE_VAL_JSON.write_text(json.dumps(smoke_val, ensure_ascii=False), encoding="utf-8")
    print_json({
        "train_examples": len(smoke_train),
        "train_tasks": {task: len(rows) for task, rows in smoke_train_by_task.items()},
        "validation_examples": len(smoke_val),
        "validation_tasks": {task: len(rows) for task, rows in smoke_val_by_task.items()},
    })




[STAGE 14] Create task-stratified smoke-test manifests
[START] 2026-09-08T19:31:05.877134+00:00
{
  "train_examples": 40,
  "train_tasks": {
    "food_classification": 8,
    "produce_variety_classification": 8,
    "nutrition_estimation": 8,
    "full_food_analysis": 8,
    "ingredient_recognition": 8
  },
  "validation_examples": 8,
  "validation_tasks": {
    "nutrition_estimation": 2,
    "food_classification": 2,
    "ingredient_recognition": 2,
    "full_food_analysis": 2
  }
}
[STAGE 14 OK] elapsed=0.01s


In [18]:
# pyright: ignore
# 15 — Load Food-R1, apply QLoRA, run forward/backward/optimizer steps on both GPUs, then free smoke artifacts.
with stage("15", "Run the full two-GPU training smoke test"):
    SMOKE_OUTPUT = Path("/kaggle/working/thrivelens_smoke_output")
    if RUN_DUAL_GPU_SMOKE_TEST:
        if SMOKE_OUTPUT.exists():
            shutil.rmtree(SMOKE_OUTPUT)
        smoke_command = build_training_command(SMOKE_TRAIN_JSON, SMOKE_VAL_JSON, SMOKE_OUTPUT, smoke=True)
        print("[SMOKE] This first model load downloads the public Food-R1 weights; no Hugging Face login is required.")
        with live_telemetry(TELEMETRY_SECONDS):
            run_streaming(
                smoke_command,
                cwd=TRAINER_ROOT,
                env=training_env,
                log_path=WORK / "dual_gpu_smoke_test.log",
                label="DUAL-GPU-SMOKE",
            )
        assert (SMOKE_OUTPUT / "adapter_config.json").exists(), "Smoke run did not save a LoRA adapter"
        assert (SMOKE_OUTPUT / "model_parameter_summary.json").exists(), "Instrumented model summary was not produced"
        smoke_summary = json.loads((SMOKE_OUTPUT / "model_parameter_summary.json").read_text(encoding="utf-8"))
        assert not smoke_summary["visual_or_merger_trainable_tensors"]
        assert smoke_summary["trainable_parameter_elements"] > 0
        assert smoke_summary["peft_configuration"]["rank"] == LORA_RANK
        assert smoke_summary["peft_configuration"]["alpha"] == LORA_ALPHA
        assert abs(smoke_summary["peft_configuration"]["dropout"] - LORA_DROPOUT) < 1e-12
        assert smoke_summary["peft_configuration"]["bias"] == LORA_BIAS
        assert smoke_summary["peft_configuration"]["target_module_count"] > 0
        assert smoke_summary["gradient_checkpointing"] == GRADIENT_CHECKPOINTING
        assert smoke_summary["gradient_checkpointing_kwargs"] == GRADIENT_CHECKPOINTING_KWARGS
        shutil.copy2(SMOKE_OUTPUT / "model_parameter_summary.json", WORK / "smoke_model_parameter_summary.json")
        print_json(smoke_summary)
        shutil.rmtree(SMOKE_OUTPUT)
        torch.cuda.empty_cache()
        print(f"[SMOKE] {SMOKE_STEPS} optimizer steps completed on {WORLD_SIZE} GPUs; runtime configuration assertions passed.")
        smoke_log_text = (WORK / "dual_gpu_smoke_test.log").read_text(encoding="utf-8", errors="replace")
        _seconds_per_step = sorted(
            float(value)
            for value in __import__("re").findall(r"([\d.]+)s/it", smoke_log_text)
            if float(value) >= 0.25
        )
        if _seconds_per_step:
            smoke_step_seconds = float(_seconds_per_step[len(_seconds_per_step) // 2])
            projected_training_hours = smoke_step_seconds * approximate_steps / 3600.0
            print(f"[TIME BUDGET] smoke steady-state {smoke_step_seconds:.2f}s/optimizer-step; full run projected ≈ {projected_training_hours:.1f} GPU-hours")
        else:
            smoke_step_seconds = None
            projected_training_hours = None
            print("[TIME BUDGET] could not parse per-step timing from the smoke log; projection skipped.")
    else:
        smoke_step_seconds = None
        projected_training_hours = None
        print("[SMOKE] Explicitly disabled in configuration. RUN_DUAL_GPU_SMOKE_TEST=False")




[STAGE 15] Run the full two-GPU training smoke test
[START] 2026-09-08T19:31:05.899040+00:00
[SMOKE] This first model load downloads the public Food-R1 weights; no Hugging Face login is required.
[DUAL-GPU-SMOKE] cwd=/kaggle/working/thrivelens_setup/training/qwen-vl-finetune
[DUAL-GPU-SMOKE] command:
/usr/local/bin/deepspeed --num_gpus 2 /kaggle/working/thrivelens_setup/training/qwen-vl-finetune/src/train/train_sft.py --deepspeed /kaggle/working/thrivelens_setup/training/qwen-vl-finetune/scripts/zero2.json --model_id zy12123/Food-R1 --data_path /kaggle/working/thrivelens_setup/smoke_train.json --image_folder /unused/thrivelens_virtual_media --remove_unused_columns False --bits 4 --double_quant True --quant_type nf4 --lora_enable True --vision_lora False --use_dora False --lora_namespan_exclude ['lm_head', 'embed_tokens', 'visual'] --lora_rank 64 --lora_alpha 16 --lora_dropout 0.05 --lora_bias none --num_lora_modules -1 --freeze_llm True --freeze_vision_tower True --freeze_merger True 

In [19]:
# pyright: ignore
# 16 — Detect a completed run or recover the newest complete compatible Kaggle checkpoint automatically.
with stage("16", "Resolve and validate checkpoint state before the expensive fixed-epoch run"):
    token = _detect_hf_token()
    if token:
        print("[SYNC] HF_TOKEN secret detected; automatic remote resume and private adapter sync are ON.")
    else:
        print("[SYNC] No HF_TOKEN found; automatic resume needs a Kaggle secret named HF_TOKEN (or ThriveLense). "
              "Without it, the manual download/re-attach resume path applies when a 12 h session crashes.")

    restore_remote_resume()
    _budget_override_path = OUTPUT_DIR / "budget_fit_override.json"
    if not _budget_override_path.exists():
        for _candidate in sorted(OUTPUT_DIR.glob("checkpoint-*/budget_fit_override.json")):
            _budget_override_path = _candidate
            break
    if _budget_override_path.exists():
        try:
            _budget_override = json.loads(_budget_override_path.read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"[TIME BUDGET] Could not read persisted auto-fit settings ({type(exc).__name__}: {exc}); defaults used.")
            _budget_override = {}
        if _budget_override:
            for _key, _value in _budget_override.items():
                globals()[_key] = _value
            print("[TIME BUDGET] Applied persisted auto-fitted settings:\n" + json.dumps(_budget_override, indent=2))
    def checkpoint_step(path: Path) -> int:
        try:
            return int(path.name.rsplit("-", 1)[1])
        except Exception:
            return -1

    def checkpoint_required_files(path: Path) -> tuple[Path, ...]:
        adapter_files = (path / "adapter_model.safetensors", path / "pytorch_model.bin")
        required = [path / "trainer_state.json", path / "training_args.bin"]
        required.append(next((candidate for candidate in adapter_files if candidate.exists()), adapter_files[0]))
        return tuple(required)

    def checkpoint_is_complete(path: Path) -> bool:
        required = checkpoint_required_files(path)
        missing = [str(item.name) for item in required if not item.exists()]
        if missing:
            print(f"[RESUME] Rejecting incomplete checkpoint {path}: missing {missing}")
            return False
        try:
            state = json.loads((path / "trainer_state.json").read_text(encoding="utf-8"))
            if int(state.get("global_step") or 0) <= 0:
                print(f"[RESUME] Rejecting checkpoint {path}: global_step is missing or zero")
                return False
            torch.load(path / "training_args.bin", map_location="cpu", weights_only=False)
        except Exception as exc:
            print(f"[RESUME] Rejecting unreadable checkpoint {path}: {type(exc).__name__}: {exc}")
            return False
        return True

    resume_contract = {
        "num_train_epochs": float(EPOCHS), "per_device_train_batch_size": MICRO_BATCH,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS, "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO, "weight_decay": WEIGHT_DECAY, "max_grad_norm": MAX_GRAD_NORM,
        "optim": OPTIMIZER, "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "lora_bias": LORA_BIAS, "num_lora_modules": -1,
        "lora_namespan_exclude": list(LORA_EXCLUDED_NAME_SPANS),
        "lora_enable": LORA_ENABLE, "vision_lora": VISION_LORA, "use_dora": USE_DORA,
        "freeze_llm": FREEZE_LLM, "freeze_vision_tower": FREEZE_VISION_TOWER,
        "freeze_merger": FREEZE_MERGER, "unfreeze_topk_llm": UNFREEZE_TOPK_LLM,
        "unfreeze_topk_vision": UNFREEZE_TOPK_VISION,
        "bits": 4 if LOAD_IN_4BIT else 16, "double_quant": BNB_4BIT_USE_DOUBLE_QUANT,
        "quant_type": BNB_4BIT_QUANT_TYPE, "fp16": FP16, "bf16": BF16, "tf32": TF32,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "gradient_checkpointing_kwargs": GRADIENT_CHECKPOINTING_KWARGS,
        "max_seq_length": MAX_SEQUENCE_LENGTH,
        "image_min_pixels": MIN_IMAGE_PIXELS, "image_max_pixels": MAX_IMAGE_PIXELS,
        "save_steps": SAVE_STEPS, "save_total_limit": SAVE_TOTAL_LIMIT,
        "eval_strategy": EVAL_STRATEGY, "save_strategy": SAVE_STRATEGY,
        "dataloader_num_workers": DATALOADER_WORKERS, "dataloader_prefetch_factor": DATALOADER_PREFETCH_FACTOR,
        "ddp_find_unused_parameters": DDP_FIND_UNUSED_PARAMETERS, "ddp_timeout": DDP_TIMEOUT_SECONDS,
        "remove_unused_columns": REMOVE_UNUSED_COLUMNS, "load_best_model_at_end": LOAD_BEST_MODEL_AT_END,
    }

    def normalized_argument(value):
        return value.value if hasattr(value, "value") else value

    def checkpoint_is_compatible(path: Path) -> bool:
        if not checkpoint_is_complete(path):
            return False
        try:
            saved_arguments = torch.load(path / "training_args.bin", map_location="cpu", weights_only=False)
        except Exception as exc:
            print(f"[RESUME] Rejecting unreadable training_args.bin in {path}: {exc}")
            return False
        mismatches = {}
        for name, expected in resume_contract.items():
            actual = normalized_argument(getattr(saved_arguments, name, None))
            if actual is None and name == "eval_strategy":
                actual = normalized_argument(getattr(saved_arguments, "evaluation_strategy", None))
            if isinstance(expected, float) and isinstance(actual, (int, float)):
                matches = math.isclose(float(actual), expected, rel_tol=1e-9, abs_tol=1e-12)
            else:
                matches = actual == expected
            if not matches:
                mismatches[name] = {"expected": expected, "actual": actual}
        if mismatches:
            print(f"[RESUME] Rejecting incompatible checkpoint: {path}")
            print_json(mismatches)
            return False
        print(f"[RESUME] Compatibility and completeness verified for {path}")
        return True

    discovered_local_checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"), key=checkpoint_step)
    local_checkpoints = [path for path in discovered_local_checkpoints if checkpoint_is_compatible(path)]
    incomplete_local = [path for path in discovered_local_checkpoints if path not in local_checkpoints]
    for path in incomplete_local:
        quarantine = path.with_name(path.name + ".invalid")
        if not quarantine.exists():
            path.rename(quarantine)
            print(f"[RESUME] Quarantined invalid checkpoint: {quarantine}")
    final_state_path = OUTPUT_DIR / "trainer_state.json"
    final_adapter_path = OUTPUT_DIR / "adapter_config.json"
    TRAIN_REQUIRED = True
    if final_state_path.exists() and final_adapter_path.exists():
        existing_state = json.loads(final_state_path.read_text(encoding="utf-8"))
        if float(existing_state.get("epoch") or 0) >= EPOCHS and local_checkpoints:
            TRAIN_REQUIRED = False
            print(f"[RESUME] A complete {existing_state['epoch']}-epoch adapter already exists; training will not duplicate it.")

    if TRAIN_REQUIRED and not local_checkpoints:
        attached_checkpoints = []
        for candidate in Path("/kaggle/input").rglob("checkpoint-*"):
            if candidate.is_dir() and "thrivelens_foodr1_qlora" in str(candidate).lower() and checkpoint_is_compatible(candidate):
                attached_checkpoints.append(candidate)
        attached_checkpoints.sort(key=checkpoint_step)
        if attached_checkpoints:
            latest_attached = attached_checkpoints[-1]
            destination = OUTPUT_DIR / latest_attached.name
            print(f"[RESUME] Importing verified attached checkpoint {latest_attached} -> {destination}")
            shutil.copytree(latest_attached, destination, dirs_exist_ok=True)
            assert checkpoint_is_compatible(destination), "Copied attached checkpoint failed post-copy validation"
            local_checkpoints = [destination]

    if local_checkpoints:
        latest = local_checkpoints[-1]
        checkpoint_state = json.loads((latest / "trainer_state.json").read_text(encoding="utf-8"))
        resume_checkpoint_path = latest
        resume_step = int(checkpoint_state.get("global_step") or 0)
        print_json({
            "resume_checkpoint": str(latest),
            "global_step": checkpoint_state.get("global_step"),
            "epoch": checkpoint_state.get("epoch"),
            "retained_checkpoint_count": len(local_checkpoints),
        })
    else:
        resume_checkpoint_path = None
        resume_step = 0
        print("[RESUME] No complete compatible checkpoint found; the full run will start from the Food-R1 base model.")
    print(f"[RESUME] TRAIN_REQUIRED={TRAIN_REQUIRED}")


[STAGE 16] Resolve and validate checkpoint state before the expensive fixed-epoch run
[START] 2026-09-08T19:35:08.441856+00:00
[SYNC] HF_TOKEN secret detected; automatic remote resume and private adapter sync are ON.
[SYNC] HF_TOKEN secret detected; automatic remote resume and private adapter sync are ON.
[SYNC] HF_TOKEN secret detected; automatic remote resume and private adapter sync are ON.
[SYNC] No remote resume repo yet (ntatemasoabi/thrivelens-foodr1-resume); starting from scratch.
[RESUME] No complete compatible checkpoint found; the full run will start from the Food-R1 base model.
[RESUME] TRAIN_REQUIRED=True
[STAGE 16 OK] elapsed=392.15s


## Full fixed-epoch fine-tuning

This is the long-running cell. It streams every DeepSpeed/Hugging Face log line to the console and to `full_training.log`. A separate telemetry line reports both GPUs and host resource use every 60 seconds. Validation runs after each epoch. Recoverable optimizer/model checkpoints are written every 250 optimizer steps and the newest two are retained. Re-run all cells after an interruption; the pinned trainer automatically resumes the newest compatible local or attached checkpoint. If an `HF_TOKEN` Kaggle secret is attached, every completed checkpoint is also mirrored to your private Hugging Face repo the moment it is saved, and the next run restores it automatically - no manual download or re-attach needed. Without a token, the manual resume flow stays available. No metric can stop the requested one epoch.

In [ ]:
# pyright: ignore
# 17 — Full one-epoch dual-T4 fine-tuning with a HARD wall-clock ceiling (<= 20 GPU-hours).
#
# Every fix from the field is wired into this cell:
#   * OOM fix: micro_batch is ALWAYS 1 (a single-sample micro-step halves the backward activation
#     peak that crashed the micro_batch=2 run at ~14.8 GiB on the 15 GiB T4s). grad_accum stays 8 so
#     the global batch remains 16; PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True stops the
#     allocator-fragmentation class of OOM; an nvidia-smi VRAM pre-flight runs before every launch.
#   * Stale-fit fix: the persisted budget_fit_override.json is NEVER trusted blindly. It may hold the
#     crashing micro_batch=2 config or an outdated pixel/seq fit, so it is discarded and recomputed
#     every launch.
#   * TIME-BUDGET fix (the reason this cell was rewritten): the old fitter PROJECTED pixels/seq from
#     stale arithmetic and said ~27.7 GPU-hours when reality was ~35 h. Heuristic projections are
#     unreliable on this hardware, so this cell now MEASURES the candidate configuration with a
#     short 3-optimizer-step benchmark on the actual dual-T4 pair, then enforces a hard ceiling:
#        projected = measured s/it  x  total steps  x  1.20 (real-data/save-burst safety)
#                   + checkpoint save+mirror overhead (90 s per checkpoint)
#     Candidates are tried in quality order (pixel richness first, all 242,756 rows). If no tier at
#     full data fits the ceiling, the fitter deterministically caps the training rows (task-stratified,
#     seed fixed) so the measured speed provably lands inside the ceiling. Pre-flight benchmarking
#     shares the 20 h budget and is capped at ~1 h; the training projection itself is <= 19 h.
#   * Resume fix: optimizer/model checkpoints keep saving every SAVE_STEPS and are mirrored to the
#     private HF repo the moment they complete; a re-run of the notebook cells resumes the newest
#     compatible checkpoint automatically (see Stage 16).
with stage("17", "Fine-tune Food-R1 within a hard 20 GPU-hour ceiling (measured benchmark)"):
    if TRAIN_REQUIRED:
        print("[TRAIN] Starting/resuming the fixed-epoch run inside a hard wall-clock ceiling.")
        print_json({
            "world_size": WORLD_SIZE,
            "global_batch_size": GLOBAL_BATCH_SIZE,
            "epochs": EPOCHS,
            "expected_approximate_optimizer_steps": approximate_steps,
            "checkpoint_interval": SAVE_STEPS,
            "validation_interval": "one complete pass per epoch",
            "early_stopping": EARLY_STOPPING,
            "hard_ceiling_gpu_hours": 20.0,
        })

        import re as _re

        # ------------------------- budget accounting -------------------------
        # User mandate: the WHOLE notebook (pre-flight benchmark + training) must complete within
        # 8 GPU-hours. Training is guarded to <= 6.5 h projected (safety included); pre-flight,
        # epoch evaluation, and held-out generation share the leftover ~1.5 h. These are enforced
        # against MEASURED speed, not guesses.
        _CEILING_HOURS = 7.5
        _TRAIN_TARGET_HOURS = 6.5
        _PROJECTION_SAFETY = 1.20
        _CKPT_SAVE_SYNC_HOURS_PER_CHECKPOINT = 90.0 / 3600.0

        # Never re-apply the stale persisted fit; recompute from a measured benchmark every launch.
        _budget_override_path = OUTPUT_DIR / "budget_fit_override.json"
        if _budget_override_path.exists():
            print("[TIME BUDGET] Discarding stale persisted-fitted settings; the fit is recomputed from a measured benchmark every launch.")
            _budget_override_path.unlink()

        # Work only on the most RECENT run inside full_training.log (it appends across runs).
        # Informational only — the measured on-device benchmark below is the source of truth.
        _log_path = OUTPUT_DIR / "full_training.log"

        def _latest_run_text() -> str | None:
            if not _log_path.exists():
                return None
            _text = _log_path.read_text(encoding="utf-8", errors="replace")
            _starts = list(_re.finditer(r"^cmd = ", _text, flags=_re.MULTILINE))
            return _text[_starts[-1].start():] if _starts else _text

        _t_prior = None
        _latest_run = _latest_run_text()
        if _latest_run:
            try:
                _prior_values = sorted(
                    float(value) for value in _re.findall(r"([\d.]+)s/it", _latest_run) if float(value) >= 0.25
                )
                if _prior_values:
                    _t_prior = float(_prior_values[len(_prior_values) // 2])
                    print(f"[TIME BUDGET] previous-run median throughput (informational): {_t_prior:.2f} s/optimizer-step.")
            except Exception:
                pass
        if _t_prior is None:
            print("[TIME BUDGET] no previous-run timing found; the on-device benchmark below provides the ground truth.")

        # Real training-row count drives the step math (not a hard-coded constant).
        _train_json = SFT_DIR / "train.json"
        _train_capped_json = SFT_DIR / "train_capped.json"
        try:
            _full_rows = json.loads(_train_json.read_text(encoding="utf-8"))
            _train_rows = len(_full_rows)
        except Exception:
            _full_rows = None
            _train_rows = TRAIN_ROWS_EXPECTED
        _total_steps = max(math.ceil(_train_rows / GLOBAL_BATCH_SIZE), 1)
        print(f"[TIME BUDGET] train rows={_train_rows:,} -> {_total_steps:,} optimizer steps at global batch {GLOBAL_BATCH_SIZE}.")

        # --------------------- measured benchmark fitter ---------------------
        _candidate_pixels = [25600, 14336, 9216, 4096]
        _candidate_seq = 192
        _benchmark_steps = 3
        _env_bench = {
            **training_env,
            "THRIVELENS_FREE_BASE_CACHE_AFTER_LOAD": "1",
            "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        }

        def _mutate_globals(**patch) -> None:
            for _name, _value in patch.items():
                globals()[_name] = _value

        def _benchmark_step_seconds(pixels: int) -> float | None:
            _smoke_out = WORK / f"benchmark_{pixels}px"
            _smoke_log = WORK / f"benchmark_{pixels}px.log"
            if _smoke_out.exists():
                shutil.rmtree(_smoke_out, ignore_errors=True)
            _mutate_globals(
                MAX_IMAGE_PIXELS=int(pixels),
                MIN_IMAGE_PIXELS=min(MIN_IMAGE_PIXELS, int(pixels)),
                MAX_SEQUENCE_LENGTH=_candidate_seq,
                LORA_RANK=16,
                MICRO_BATCH=1,
                GRADIENT_ACCUMULATION_STEPS=8,
                GLOBAL_BATCH_SIZE=1 * 8 * WORLD_SIZE,
                SMOKE_STEPS=_benchmark_steps,
            )
            _cmd = build_training_command(SMOKE_TRAIN_JSON, None, _smoke_out, smoke=True)
            print(f"[BENCHMARK] measuring steady-state throughput at img {pixels}px/seq {_candidate_seq}/r16/micro 1/ga 8 "
                  f"({_benchmark_steps} optimizer steps, 2 GPUs)...", flush=True)
            try:
                with live_telemetry(TELEMETRY_SECONDS):
                    run_streaming(_cmd, cwd=TRAINER_ROOT, env=_env_bench, log_path=_smoke_log, label=f"BENCHMARK-{pixels}px")
            except Exception as exc:
                print(f"[BENCHMARK] benchmark run failed ({type(exc).__name__}: {exc}); treating the tier as unmeasured.")
                shutil.rmtree(_smoke_out, ignore_errors=True)
                torch.cuda.empty_cache()
                return None
            try:
                _text = _smoke_log.read_text(encoding="utf-8", errors="replace")
                _values = sorted(float(value) for value in _re.findall(r"([\d.]+)s/it", _text) if float(value) >= 0.25)
            except Exception:
                _values = []
            shutil.rmtree(_smoke_out, ignore_errors=True)
            torch.cuda.empty_cache()
            if not _values:
                print(f"[BENCHMARK] could not parse per-step timing from the {pixels}px benchmark log.")
                return None
            return float(_values[len(_values) // 2])

        def _projected_hours(seconds_per_step: float, steps: int) -> float:
            _ckpt_overhead = max(0.0, (steps // SAVE_STEPS) * _CKPT_SAVE_SYNC_HOURS_PER_CHECKPOINT)
            return (seconds_per_step * steps / 3600.0) * _PROJECTION_SAFETY + _ckpt_overhead

        _benchmark_report = {}
        _chosen_pixels = None
        _chosen_sit = None
        _chosen_hours = None
        # --------------------- automatic resume fast path ---------------------
        # When a compatible checkpoint exists (restored remotely by SYNC-AUX or staged by
        # Stage 16), the pixel tier, measured throughput, and data path were already settled
        # in the interrupted session. Re-benchmarking after every 12 h crash burns 15-25
        # restart minutes for zero new information, so reuse the persisted auto-fit and
        # project only the remaining optimizer steps.
        _resume_cfg: dict | None = None
        if resume_checkpoint_path is not None:
            _saved_override_candidates = [
                resume_checkpoint_path / "budget_fit_override.json",
                OUTPUT_DIR / "budget_fit_override.json",
            ]
            for _saved_override in _saved_override_candidates:
                if not _saved_override.exists():
                    continue
                try:
                    _candidate_cfg = json.loads(_saved_override.read_text(encoding="utf-8"))
                except Exception:
                    _candidate_cfg = {}
                if float(_candidate_cfg.get("measured_seconds_per_step") or 0.0) > 0.0:
                    _resume_cfg = _candidate_cfg
                    break
        if _resume_cfg is not None:
            _resume_sit = float(_resume_cfg["measured_seconds_per_step"])
            _resume_batch = int(_resume_cfg.get("GLOBAL_BATCH_SIZE") or GLOBAL_BATCH_SIZE)
            _resume_data_path = Path(_resume_cfg.get("training_data_path") or _train_json)
            _resume_data_valid = _resume_data_path.exists()
            _resume_cap = _resume_cfg.get("TRAIN_ROW_CAP")
            _resume_pixels = int(_resume_cfg.get("MAX_IMAGE_PIXELS", _candidate_pixels[0]))
            if _resume_data_valid:
                try:
                    _resume_rows = len(json.loads(_resume_data_path.read_text(encoding="utf-8")))
                except Exception:
                    _resume_rows = _train_rows
            else:
                _resume_rows = _train_rows
            _resume_total_steps = max(math.ceil(_resume_rows / _resume_batch), 1)
            _remaining_steps = max(_resume_total_steps - int(resume_step), 0)
            _projected_resume_hours = _projected_hours(_resume_sit, _remaining_steps)
            if _projected_resume_hours > _CEILING_HOURS:
                print(f"[RESUME][WARNING] saved timing projects {_projected_resume_hours:.1f}h for the remaining "
                      f"{_remaining_steps:,} steps and would cross the {_CEILING_HOURS:.0f}h ceiling; re-benchmarking fresh.")
                _resume_cfg = None
            else:
                _benchmark_report = {
                    "resume": {
                        "checkpoint": str(resume_checkpoint_path),
                        "saved_global_step": int(resume_step),
                        "remaining_steps": _remaining_steps,
                        "total_steps_at_saved_config": _resume_total_steps,
                        "measured_seconds_per_step": round(_resume_sit, 3),
                        "projected_remaining_gpu_hours": round(_projected_resume_hours, 2),
                    },
                }
                _chosen_pixels = _resume_pixels
                _chosen_sit = _resume_sit
                _chosen_hours = _projected_resume_hours
                _train_row_cap = int(_resume_cap) if _resume_cap is not None else None
                _steps_actual = _remaining_steps
                _data_json = _resume_data_path if _resume_data_valid else _train_json
                print(f"[RESUME] continuing from {resume_checkpoint_path} at step {resume_step}: "
                      f"{_remaining_steps:,} optimiser steps remain at ~{_resume_sit:.2f} s/it "
                      f"(saved timing; no re-benchmark on this restart).")
                print(f"[RESUME] projected {_projected_resume_hours:.1f} GPU-hours for the remaining training "
                      f"(hard ceiling {_CEILING_HOURS:.0f}h).")
        if _resume_cfg is None:

          # 1) Prefer the richest pixel tier that fits the FULL dataset inside the training target.
          for _px in _candidate_pixels:
              _sit = _benchmark_step_seconds(_px)
              _benchmark_report[str(_px)] = {"measured_s_it": _sit}
              if _sit is None:
                  continue
              _proj = _projected_hours(_sit, _total_steps)
              _benchmark_report[str(_px)]["projected_full_data_hours"] = round(_proj, 2)
              _verdict = "OK" if _proj <= _TRAIN_TARGET_HOURS else "over budget"
              print(f"[TIME BUDGET] img {_px}px/seq {_candidate_seq} measured {_sit:.2f} s/it -> full-data projection "
                    f"{_proj:.1f} GPU-hours [{_verdict}].")
              if _proj <= _TRAIN_TARGET_HOURS:
                  _chosen_pixels, _chosen_sit, _chosen_hours = _px, _sit, _proj
                  break
              if _px == _candidate_pixels[-1]:
                  _chosen_sit = _sit
                  _chosen_hours = _proj

          # 2) Guaranteed fallback: if no pixel tier at full data fits, deterministically cap the
          #    training rows at the richest tier until the measured speed lands inside the ceiling.
          _train_row_cap = None
          if _chosen_pixels is None:
              _fallback_pixels = _candidate_pixels[0]
              _fallback_sit = (_benchmark_report.get(str(_fallback_pixels)) or {}).get("measured_s_it")
              if _fallback_sit is None:
                  _fallback_sit = _benchmark_step_seconds(_fallback_pixels)
                  _benchmark_report.setdefault(str(_fallback_pixels), {})["measured_s_it"] = _fallback_sit
              if _fallback_sit is None:
                  _fallback_sit = 8.4  # last-resort constant from the observed live run; the cap math still guarantees the ceiling
              _lo, _hi = 1, _total_steps
              while _lo < _hi:
                  _mid = (_lo + _hi + 1) // 2
                  if _projected_hours(_fallback_sit, _mid) <= _TRAIN_TARGET_HOURS:
                      _lo = _mid
                  else:
                      _hi = _mid - 1
              _cap_steps = _lo
              _cap_rows = min(_train_rows, _cap_steps * GLOBAL_BATCH_SIZE)
              if _full_rows is not None:
                  _by_task = defaultdict(list)
                  for _row in _full_rows:
                      _by_task[_row.get("task", "other")].append(_row)
                  _capped_by_task = []
                  for _task, _task_rows in sorted(_by_task.items()):
                      _keep = max(1, int(round(len(_task_rows) * _cap_rows / max(_train_rows, 1))))
                      _ordered = sorted(
                          enumerate(_task_rows),
                          key=lambda pair: hashlib.sha256(f"{_task}::{pair[0]}::{SEED}".encode("utf-8")).hexdigest(),
                      )
                      _capped_by_task.append([row for _, row in _ordered[:_keep]])
                  _capped_rows = [row for group in _capped_by_task for row in group]
                  _train_capped_json.write_text(json.dumps(_capped_rows, ensure_ascii=False), encoding="utf-8")
                  _train_row_cap = len(_capped_rows)
              else:
                  _train_row_cap = _cap_rows
              _steps_actual = max(math.ceil(_train_row_cap / GLOBAL_BATCH_SIZE), 1)
              _chosen_pixels = _fallback_pixels
              _chosen_hours = _projected_hours(_fallback_sit, _steps_actual)
              print(f"[TIME BUDGET][WARNING] no pixel tier at full data fit the ceiling; capping training rows to "
                    f"{_train_row_cap:,} (={_steps_actual:,} steps, task-stratified, deterministic) at img {_fallback_pixels}px so "
                    f"the measured {_fallback_sit:.2f} s/it lands inside the budget.")
          else:
              _steps_actual = _total_steps

          _data_json = _train_capped_json if _train_row_cap is not None else _train_json
          _final_hours = _chosen_hours if _chosen_hours is not None else _projected_hours(_chosen_sit or 8.4, _steps_actual)
          assert _final_hours <= _CEILING_HOURS, f"Projection {_final_hours:.1f}h crossed the hard {_CEILING_HOURS:.0f}h ceiling; this must never happen."
        else:
            _final_hours = _chosen_hours
        assert _final_hours <= _CEILING_HOURS, f"Projection {_final_hours:.1f}h crossed the hard {_CEILING_HOURS:.0f}h ceiling; this must never happen."

        # Commit the chosen config as globals and write the auditable override for later stages/sessions.
        _mutate_globals(
            MAX_IMAGE_PIXELS=int(_chosen_pixels),
            MIN_IMAGE_PIXELS=min(MIN_IMAGE_PIXELS, int(_chosen_pixels)),
            MAX_SEQUENCE_LENGTH=_candidate_seq,
            LORA_RANK=16,
            MICRO_BATCH=1,
            GRADIENT_ACCUMULATION_STEPS=8,
            GLOBAL_BATCH_SIZE=1 * 8 * WORLD_SIZE,
        )
        _budget_override = {
            "MAX_IMAGE_PIXELS": MAX_IMAGE_PIXELS,
            "MIN_IMAGE_PIXELS": MIN_IMAGE_PIXELS,
            "MAX_SEQUENCE_LENGTH": MAX_SEQUENCE_LENGTH,
            "LORA_RANK": LORA_RANK,
            "MICRO_BATCH": MICRO_BATCH,
            "GRADIENT_ACCUMULATION_STEPS": GRADIENT_ACCUMULATION_STEPS,
            "GLOBAL_BATCH_SIZE": GLOBAL_BATCH_SIZE,
            "TRAIN_ROW_CAP": _train_row_cap if _train_row_cap is not None else None,
            "training_data_path": str(_data_json),
            "measured_seconds_per_step": round(_chosen_sit if _chosen_sit is not None else 8.4, 3),
            "projected_gpu_hours": round(_final_hours, 2),
            "hard_ceiling_gpu_hours": _CEILING_HOURS,
            "projection_safety": _PROJECTION_SAFETY,
        }
        _budget_override_path.write_text(json.dumps(_budget_override, indent=2), encoding="utf-8")
        print(f"[TIME BUDGET] selected config: img {MAX_IMAGE_PIXELS}px/seq {MAX_SEQUENCE_LENGTH}/r{LORA_RANK}/"
              f"micro {MICRO_BATCH}/ga {GRADIENT_ACCUMULATION_STEPS}"
              + (f" with {_train_row_cap:,} capped train rows" if _train_row_cap is not None else "")
              + ".")
        print(f"[TIME BUDGET] final projected training consumption ~= {_final_hours:.1f} GPU-hours "
              f"(measured {(_chosen_sit if _chosen_sit is not None else 8.4):.2f} s/it x {_steps_actual:,} steps x "
              f"{_PROJECTION_SAFETY:.2f} safety + checkpoint/sync overhead) against the {_CEILING_HOURS:.0f} GPU-hour "
              f"ceiling (pre-flight benchmark shares the leftover budget).")

        # Rebuild the real training command on the chosen config and prove it matches the contract.
        FULL_COMMAND = build_training_command(_data_json, SFT_DIR / "val.json", OUTPUT_DIR, smoke=False)
        for _option, _expected in {
            "--max_seq_length": str(MAX_SEQUENCE_LENGTH),
            "--image_max_pixels": str(MAX_IMAGE_PIXELS),
            "--lora_rank": str(LORA_RANK),
            "--gradient_accumulation_steps": str(GRADIENT_ACCUMULATION_STEPS),
            "--per_device_train_batch_size": str(MICRO_BATCH),
        }.items():
            assert command_value(FULL_COMMAND, _option) == _expected, f"Rebuilt command drifted at {_option}"
        (OUTPUT_DIR / "full_training_command.json").write_text(json.dumps(FULL_COMMAND, indent=2), encoding="utf-8")
        (OUTPUT_DIR / "full_training_command.sh").write_text(
            "#!/usr/bin/env bash\nset -euo pipefail\n" + shlex.join(FULL_COMMAND) + "\n", encoding="utf-8"
        )
        (OUTPUT_DIR / "benchmark_timing.json").write_text(json.dumps(_benchmark_report, indent=2), encoding="utf-8")

        _train_env = {
            **training_env,
            "THRIVELENS_FREE_BASE_CACHE_AFTER_LOAD": "1",
            "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        }
        print("[TRAIN] /kaggle/working free space before launch:")
        subprocess.run(["df", "-h", "/kaggle/working"], check=True)
        _smi = subprocess.run(["nvidia-smi", "--query-gpu=memory.free,memory.total", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True)
        if _smi.returncode == 0 and _smi.stdout.strip():
            for _i, _line in enumerate(_smi.stdout.strip().splitlines()):
                _free_mib, _total_mib = (part.strip() for part in _line.split(","))
                print(f"[TRAIN] GPU {_i} pre-flight VRAM: {_free_mib} MiB free / {_total_mib} MiB total")
            _lowest_free = min(int(line.split(",")[0]) for line in _smi.stdout.strip().splitlines())
            if _lowest_free < 2048:
                print("[TRAIN][WARNING] A GPU is nearly full before launch; check for stray processes and rerun if the launch fails.")
        _resume_watcher = ResumeSyncWatcher()
        _resume_watcher.start()
        try:
            with live_telemetry(TELEMETRY_SECONDS):
                run_streaming(
                    FULL_COMMAND,
                    cwd=TRAINER_ROOT,
                    env=_train_env,
                    log_path=OUTPUT_DIR / "full_training.log",
                    label="THRIVELENS-TRAIN",
                )
        finally:
            _resume_watcher.stop()
            _resume_watcher.final_sweep()
            print("[SYNC] Checkpoint watcher stopped; the newest recoverable checkpoint was mirrored remotely.")
    else:
        print("[TRAIN] Complete adapter detected. Training was safely skipped; analysis and held-out evaluation continue.")

    assert (OUTPUT_DIR / "adapter_config.json").exists(), "Final LoRA adapter configuration is missing"
    assert (OUTPUT_DIR / "trainer_state.json").exists(), "Final Trainer state is missing"
    print(f"[TRAIN] Adapter output directory: {OUTPUT_DIR}")


In [ ]:
# pyright: ignore
# 18 — Completion assertions, exact reproducibility manifest, and numerical sanity checks.
with stage("18", "Prove that fixed-epoch training completed correctly"):
    state = json.loads((OUTPUT_DIR / "trainer_state.json").read_text(encoding="utf-8"))
    completed_epoch = float(state.get("epoch") or 0)
    assert completed_epoch >= EPOCHS - 1e-6, f"Training ended at epoch {completed_epoch}, below requested {EPOCHS}"
    assert int(state.get("global_step") or 0) > 0
    log_history = state.get("log_history", [])
    numeric_loss_values = [float(item["loss"]) for item in log_history if "loss" in item]
    eval_loss_values = [float(item["eval_loss"]) for item in log_history if "eval_loss" in item]
    assert numeric_loss_values, "Trainer state contains no logged training loss"
    assert eval_loss_values, "Trainer state contains no validation loss"
    assert all(math.isfinite(value) for value in numeric_loss_values + eval_loss_values), "Non-finite loss detected"

    adapter_files = sorted(
        str(path.relative_to(OUTPUT_DIR))
        for path in OUTPUT_DIR.iterdir()
        if path.is_file()
    )
    run_manifest = {
        "project": "ThriveLens 2026",
        "base_model": MODEL_ID,
        "model_license_source": "public Hugging Face model repository",
        "dataset_mode": DATA_MODE,
        "dataset_archive_sha256": archive_sha256,
        "dataset_manifest_version": manifest.get("version"),
        "trainer_commit": expected_commit,
        "software_versions": version_report,
        "training_method": "supervised NF4 QLoRA with DeepSpeed ZeRO-2 DDP",
        "epochs_requested": EPOCHS,
        "epochs_completed": completed_epoch,
        "global_steps": int(state["global_step"]),
        "world_size": WORLD_SIZE,
        "global_effective_batch_size": GLOBAL_BATCH_SIZE,
        "micro_batch_per_gpu": MICRO_BATCH,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "optimizer": OPTIMIZER,
        "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "max_gradient_norm": MAX_GRAD_NORM,
        "quantization": {"load_in_4bit": LOAD_IN_4BIT, "type": BNB_4BIT_QUANT_TYPE, "double_quant": BNB_4BIT_USE_DOUBLE_QUANT, "compute_dtype": BNB_4BIT_COMPUTE_DTYPE},
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
        "lora_excluded_name_spans": LORA_EXCLUDED_NAME_SPANS,
        "lora_bias": LORA_BIAS,
        "precision": {"fp16": FP16, "bf16": BF16, "tf32": TF32},
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "gradient_checkpointing_kwargs": GRADIENT_CHECKPOINTING_KWARGS,
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "image_pixel_range": [MIN_IMAGE_PIXELS, MAX_IMAGE_PIXELS],
        "checkpoint_policy": {"strategy": SAVE_STRATEGY, "steps": SAVE_STEPS, "total_limit": SAVE_TOTAL_LIMIT},
        "vision_tower_frozen": FREEZE_VISION_TOWER,
        "merger_frozen": FREEZE_MERGER,
        "early_stopping": EARLY_STOPPING,
        "load_best_model_at_end": LOAD_BEST_MODEL_AT_END,
        "final_files": adapter_files,
        "completed_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    }
    (OUTPUT_DIR / "thrivelens_run_manifest.json").write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
    print_json(run_manifest)
    print("[TRAIN] Last five Trainer log entries:")
    print_json(log_history[-5:])



In [ ]:
# pyright: ignore
# 19 — Training/validation curves and an explicit overfitting/generalization diagnostic.
with stage("19", "Generate loss, learning-rate, gradient, and generalization reports"):
    import matplotlib.pyplot as plt
    import pandas as pd

    log_frame = pd.DataFrame(log_history)
    log_csv = OUTPUT_DIR / "trainer_log_history.csv"
    log_frame.to_csv(log_csv, index=False)
    train_frame = log_frame[log_frame["loss"].notna()].copy() if "loss" in log_frame else pd.DataFrame()
    eval_frame = log_frame[log_frame["eval_loss"].notna()].copy() if "eval_loss" in log_frame else pd.DataFrame()
    assert not train_frame.empty and not eval_frame.empty

    figure, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes[0, 0].plot(train_frame["step"], train_frame["loss"], linewidth=1.0, color="#2563eb")
    axes[0, 0].set(title="Training loss", xlabel="Optimizer step", ylabel="Loss")
    axes[0, 0].grid(alpha=0.25)
    axes[0, 1].plot(eval_frame["epoch"], eval_frame["eval_loss"], marker="o", linewidth=2.0, color="#dc2626")
    axes[0, 1].set(title="Complete validation loss by epoch", xlabel="Epoch", ylabel="Validation loss")
    axes[0, 1].grid(alpha=0.25)
    if "learning_rate" in train_frame:
        axes[1, 0].plot(train_frame["step"], train_frame["learning_rate"], linewidth=1.0, color="#059669")
    axes[1, 0].set(title="Learning-rate schedule", xlabel="Optimizer step", ylabel="Learning rate")
    axes[1, 0].grid(alpha=0.25)
    if "grad_norm" in train_frame:
        finite_grad = train_frame[train_frame["grad_norm"].notna()]
        axes[1, 1].plot(finite_grad["step"], finite_grad["grad_norm"], linewidth=0.8, color="#7c3aed")
        axes[1, 1].axhline(MAX_GRAD_NORM, linestyle="--", color="#111827", label="clip threshold")
        axes[1, 1].legend()
    axes[1, 1].set(title="Gradient norm", xlabel="Optimizer step", ylabel="Norm")
    axes[1, 1].grid(alpha=0.25)
    figure.suptitle("ThriveLens Food-R1 fine-tuning diagnostics", fontsize=16)
    figure.tight_layout()
    curve_path = OUTPUT_DIR / "training_diagnostics.png"
    figure.savefig(curve_path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(figure)

    best_eval_row = eval_frame.loc[eval_frame["eval_loss"].astype(float).idxmin()]
    final_eval = float(eval_frame.iloc[-1]["eval_loss"])
    best_eval = float(best_eval_row["eval_loss"])
    diagnostics = {
        "training_loss_first_logged": float(train_frame.iloc[0]["loss"]),
        "training_loss_last_logged": float(train_frame.iloc[-1]["loss"]),
        "validation_evaluations": len(eval_frame),
        "best_validation_loss": best_eval,
        "best_validation_epoch_for_reporting_only": float(best_eval_row["epoch"]),
        "final_validation_loss": final_eval,
        "final_vs_best_validation_loss_percent": 100.0 * (final_eval - best_eval) / max(abs(best_eval), 1e-12),
        "overfitting_warning": bool(final_eval > best_eval * 1.02),
        "action_taken": "reported only; fixed epochs were not stopped or shortened",
    }
    (OUTPUT_DIR / "generalization_diagnostics.json").write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")
    print_json(diagnostics)
    if diagnostics["overfitting_warning"]:
        print("[GENERALIZATION WARNING] Final validation loss is >2% above the best observed value. Training still completed all fixed epochs as requested.")
    else:
        print("[GENERALIZATION] No >2% final-vs-best validation-loss regression was observed.")



In [ ]:
# pyright: ignore
# 20 — Inspect every saved LoRA tensor and reconcile it with the live model summary.
with stage("20", "Validate the final adapter structure and parameter count"):
    from safetensors import safe_open

    adapter_weights = OUTPUT_DIR / "adapter_model.safetensors"
    assert adapter_weights.exists(), f"Missing {adapter_weights}"
    tensor_summaries = []
    adapter_elements = 0
    with safe_open(adapter_weights, framework="pt", device="cpu") as handle:
        keys = list(handle.keys())
        for key in keys:
            tensor = handle.get_tensor(key)
            elements = tensor.numel()
            adapter_elements += elements
            tensor_summaries.append({"name": key, "shape": list(tensor.shape), "dtype": str(tensor.dtype), "elements": elements})
    assert keys and adapter_elements > 0

    parameter_summary_path = OUTPUT_DIR / "model_parameter_summary.json"
    assert parameter_summary_path.exists()
    parameter_summary = json.loads(parameter_summary_path.read_text(encoding="utf-8"))
    assert not parameter_summary["visual_or_merger_trainable_tensors"]
    saved_adapter_config = json.loads((OUTPUT_DIR / "adapter_config.json").read_text(encoding="utf-8"))
    assert saved_adapter_config["r"] == LORA_RANK
    assert saved_adapter_config["lora_alpha"] == LORA_ALPHA
    assert abs(saved_adapter_config["lora_dropout"] - LORA_DROPOUT) < 1e-12
    assert saved_adapter_config["bias"] == LORA_BIAS
    assert saved_adapter_config.get("target_modules"), "Saved adapter contains no target modules"
    adapter_report = {
        "adapter_file": str(adapter_weights),
        "adapter_size_mib": adapter_weights.stat().st_size / 2**20,
        "tensor_count": len(keys),
        "tensor_elements": adapter_elements,
        "live_trainable_parameter_elements": parameter_summary["trainable_parameter_elements"],
        "trainable_percent": parameter_summary["trainable_percent"],
        "visual_or_merger_trainable_tensors": parameter_summary["visual_or_merger_trainable_tensors"],
        "saved_peft_configuration": {
            "rank": saved_adapter_config["r"], "alpha": saved_adapter_config["lora_alpha"],
            "dropout": saved_adapter_config["lora_dropout"], "bias": saved_adapter_config["bias"],
            "target_module_count": len(saved_adapter_config["target_modules"]),
        },
        "first_20_tensors": tensor_summaries[:20],
    }
    (OUTPUT_DIR / "adapter_integrity_report.json").write_text(json.dumps(adapter_report, indent=2), encoding="utf-8")
    print_json(adapter_report)



## Deterministic held-out generation and hallucination indicators

Validation loss alone is not enough for a multimodal generator. The evaluation cell launches both T4s and runs deterministic greedy decoding. The safety profile evaluates a 2,000-record diagnostic sample by default so the training session is not exposed to an additional full-session timeout; set `MAX_TEST_SAMPLES = 0` deliberately in the safety override cell only when a full 52,941-record evaluation has a separate runtime budget. Reporting includes answer-tag compliance, exact match, token F1, valid-JSON rate, ingredient-set precision/recall/F1, nutrition Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), Mean Absolute Percentage Error (MAPE), failures, throughput, and GPU memory.

In [ ]:
# pyright: ignore
# 21 — Materialize and compile the self-contained advanced distributed evaluator.
with stage("21", "Install the task-aware Food-R1 held-out evaluator"):
    EVALUATOR_SOURCE = '#!/usr/bin/env python3\n"""Distributed, task-aware held-out evaluation for ThriveLens Food-R1."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport io\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nimport traceback\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport torch\nimport torch.distributed as dist\nfrom PIL import Image\nfrom peft import PeftModel\nfrom transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration\n\n\nNUMERIC_FIELDS = ("mass_g", "total_calories", "fat_g", "carb_g", "protein_g")\nCLASSIFICATION_TASKS = {"food_classification", "produce_variety_classification"}\nJSON_TASKS = {"ingredient_recognition", "nutrition_estimation", "full_food_analysis"}\n\n\ndef normalize(text: object) -> str:\n    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()\n\n\ndef answer_block(text: str) -> tuple[str, bool]:\n    match = re.search(r"<answer>(.*?)</answer>", text or "", flags=re.I | re.S)\n    return ((match.group(1) if match else (text or "")).strip(), bool(match))\n\n\ndef token_f1(prediction: str, reference: str) -> float:\n    pred_tokens = normalize(prediction).split()\n    ref_tokens = normalize(reference).split()\n    if not pred_tokens and not ref_tokens:\n        return 1.0\n    if not pred_tokens or not ref_tokens:\n        return 0.0\n    pred_counts = defaultdict(int)\n    ref_counts = defaultdict(int)\n    for token in pred_tokens:\n        pred_counts[token] += 1\n    for token in ref_tokens:\n        ref_counts[token] += 1\n    overlap = sum(min(count, ref_counts[token]) for token, count in pred_counts.items())\n    precision = overlap / len(pred_tokens)\n    recall = overlap / len(ref_tokens)\n    return 2 * precision * recall / max(precision + recall, 1e-12)\n\n\ndef parse_json_answer(text: str) -> dict | list | None:\n    block, _ = answer_block(text)\n    candidates = [block]\n    fenced = re.search(r"```(?:json)?\\s*(.*?)```", block, flags=re.I | re.S)\n    if fenced:\n        candidates.append(fenced.group(1).strip())\n    obj = re.search(r"(\\{.*\\}|\\[.*\\])", block, flags=re.S)\n    if obj:\n        candidates.append(obj.group(1))\n    for candidate in candidates:\n        try:\n            value = json.loads(candidate)\n            if isinstance(value, (dict, list)):\n                return value\n        except (TypeError, ValueError, json.JSONDecodeError):\n            pass\n    return None\n\n\ndef ingredient_names(payload: dict | list | None) -> set[str]:\n    if payload is None:\n        return set()\n    values = payload.get("ingredients", []) if isinstance(payload, dict) else payload\n    if not isinstance(values, list):\n        return set()\n    result = set()\n    for item in values:\n        name = item.get("name", "") if isinstance(item, dict) else item\n        clean = normalize(name)\n        if clean:\n            result.add(clean)\n    return result\n\n\ndef add_ingredient_scores(scores: defaultdict[str, float], pred: set[str], ref: set[str], prefix: str) -> None:\n    true_positive = len(pred & ref)\n    precision = true_positive / max(len(pred), 1)\n    recall = true_positive / max(len(ref), 1)\n    f1 = 2 * precision * recall / max(precision + recall, 1e-12)\n    scores[f"{prefix}ingredient_precision_sum"] += precision\n    scores[f"{prefix}ingredient_recall_sum"] += recall\n    scores[f"{prefix}ingredient_f1_sum"] += f1\n    scores[f"{prefix}ingredient_examples"] += 1\n\n\ndef add_numeric_scores(scores: defaultdict[str, float], pred: dict | None, ref: dict | None, prefix: str) -> None:\n    if not isinstance(pred, dict) or not isinstance(ref, dict):\n        return\n    for field in NUMERIC_FIELDS:\n        try:\n            predicted = float(pred[field])\n            actual = float(ref[field])\n        except (KeyError, TypeError, ValueError):\n            continue\n        if not (math.isfinite(predicted) and math.isfinite(actual)):\n            continue\n        error = predicted - actual\n        scores[f"{prefix}numeric::{field}::abs"] += abs(error)\n        scores[f"{prefix}numeric::{field}::sq"] += error * error\n        scores[f"{prefix}numeric::{field}::count"] += 1\n        if abs(actual) > 1e-8:\n            scores[f"{prefix}numeric::{field}::ape"] += abs(error / actual) * 100.0\n            scores[f"{prefix}numeric::{field}::ape_count"] += 1\n\n\ndef open_media(uri: str, archive: zipfile.ZipFile | None, dataset_root: Path | None) -> Image.Image:\n    member = uri[len("zip://") :] if uri.startswith("zip://") else uri\n    member = member.lstrip("/\\\\")\n    if archive is not None:\n        raw = archive.read(member)\n        with Image.open(io.BytesIO(raw)) as image:\n            return image.convert("RGB").copy()\n    if dataset_root is None:\n        raise RuntimeError("Neither --archive nor --dataset-root was provided")\n    image_path = dataset_root / Path(member)\n    with Image.open(image_path) as image:\n        return image.convert("RGB").copy()\n\n\ndef summarize(scores: dict[str, float]) -> dict:\n    successes = int(scores.get("successes", 0))\n    requested = int(scores.get("requested", 0))\n    report = {\n        "requested_samples": requested,\n        "successful_samples": successes,\n        "failed_samples": int(scores.get("failures", 0)),\n        "failure_rate": scores.get("failures", 0) / max(requested, 1),\n        "answer_tag_compliance": scores.get("answer_tag", 0) / max(successes, 1),\n        "exact_match": scores.get("exact", 0) / max(successes, 1),\n        "token_f1": scores.get("token_f1", 0) / max(successes, 1),\n        "valid_json_rate_on_json_tasks": scores.get("valid_json", 0) / max(scores.get("json_examples", 0), 1),\n        "elapsed_seconds_sum_across_ranks": scores.get("elapsed", 0),\n        "generated_tokens": int(scores.get("generated_tokens", 0)),\n        "aggregate_generation_tokens_per_second": scores.get("generated_tokens", 0) / max(scores.get("max_elapsed", 0), 1e-9),\n        "by_task": {},\n    }\n    tasks = sorted(key.split("::", 1)[1] for key in scores if key.startswith("task_count::"))\n    for task in tasks:\n        count = scores[f"task_count::{task}"]\n        prefix = f"task::{task}::"\n        task_report = {\n            "samples": int(count),\n            "exact_match": scores.get(prefix + "exact", 0) / max(count, 1),\n            "token_f1": scores.get(prefix + "token_f1", 0) / max(count, 1),\n            "answer_tag_compliance": scores.get(prefix + "answer_tag", 0) / max(count, 1),\n        }\n        json_count = scores.get(prefix + "json_examples", 0)\n        if json_count:\n            task_report["valid_json_rate"] = scores.get(prefix + "valid_json", 0) / json_count\n        ing_count = scores.get(prefix + "ingredient_examples", 0)\n        if ing_count:\n            task_report["ingredient_set_precision"] = scores.get(prefix + "ingredient_precision_sum", 0) / ing_count\n            task_report["ingredient_set_recall"] = scores.get(prefix + "ingredient_recall_sum", 0) / ing_count\n            task_report["ingredient_set_f1"] = scores.get(prefix + "ingredient_f1_sum", 0) / ing_count\n        numeric = {}\n        for field in NUMERIC_FIELDS:\n            n = scores.get(prefix + f"numeric::{field}::count", 0)\n            if n:\n                numeric[field] = {\n                    "samples": int(n),\n                    "mae": scores[prefix + f"numeric::{field}::abs"] / n,\n                    "rmse": math.sqrt(scores[prefix + f"numeric::{field}::sq"] / n),\n                    "mape_percent": scores.get(prefix + f"numeric::{field}::ape", 0) / max(scores.get(prefix + f"numeric::{field}::ape_count", 0), 1),\n                }\n        if numeric:\n            task_report["numeric_metrics"] = numeric\n        report["by_task"][task] = task_report\n    return report\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    source = parser.add_mutually_exclusive_group(required=True)\n    source.add_argument("--archive")\n    source.add_argument("--dataset-root")\n    parser.add_argument("--test-json", required=True)\n    parser.add_argument("--adapter", required=True)\n    parser.add_argument("--model", default="zy12123/Food-R1")\n    parser.add_argument("--max-samples", type=int, default=0)\n    parser.add_argument("--max-new-tokens", type=int, default=256)\n    parser.add_argument("--min-pixels", type=int, default=256 * 256)\n    parser.add_argument("--max-pixels", type=int, default=512 * 512)\n    parser.add_argument("--seed", type=int, default=2026)\n    parser.add_argument("--progress-every", type=int, default=50)\n    parser.add_argument("--output", required=True)\n    args = parser.parse_args()\n\n    dist.init_process_group("nccl")\n    rank = dist.get_rank()\n    world = dist.get_world_size()\n    device = torch.device(f"cuda:{rank}")\n    torch.cuda.set_device(device)\n    random.seed(args.seed + rank)\n    torch.manual_seed(args.seed + rank)\n    print(f"[EVAL][rank {rank}/{world}] device={torch.cuda.get_device_name(rank)} pid={os.getpid()}", flush=True)\n\n    with open(args.test_json, encoding="utf-8") as handle:\n        all_rows = json.load(handle)\n    random.Random(args.seed).shuffle(all_rows)\n    if args.max_samples > 0:\n        all_rows = all_rows[: args.max_samples]\n    selected_count = len(all_rows)\n    rows = all_rows[rank::world]\n    print(f"[EVAL][rank {rank}] assigned={len(rows):,} selected_global={selected_count:,}", flush=True)\n\n    quantization = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_compute_dtype=torch.float16,\n    )\n    load_started = time.perf_counter()\n    model = Qwen3VLForConditionalGeneration.from_pretrained(\n        args.model,\n        torch_dtype=torch.float16,\n        quantization_config=quantization,\n        device_map={"": rank},\n        attn_implementation="sdpa",\n        trust_remote_code=True,\n    )\n    model = PeftModel.from_pretrained(model, args.adapter).eval()\n    processor = AutoProcessor.from_pretrained(\n        args.model,\n        trust_remote_code=True,\n        min_pixels=args.min_pixels,\n        max_pixels=args.max_pixels,\n    )\n    print(f"[EVAL][rank {rank}] model+adapter loaded in {time.perf_counter() - load_started:.1f}s", flush=True)\n\n    archive = zipfile.ZipFile(args.archive, "r") if args.archive else None\n    dataset_root = Path(args.dataset_root) if args.dataset_root else None\n    local: defaultdict[str, float] = defaultdict(float)\n    local["requested"] = len(rows)\n    predictions = []\n    failures = []\n    started = time.perf_counter()\n\n    for index, row in enumerate(rows, start=1):\n        sample_started = time.perf_counter()\n        task = row.get("task", "unknown")\n        try:\n            paths = row["image"] if isinstance(row["image"], list) else [row["image"]]\n            images = [open_media(path, archive, dataset_root) for path in paths]\n            prompt = row["conversations"][0]["value"].replace("<image>\\n", "").replace("<image>", "").strip()\n            content = [*[{"type": "image", "image": image} for image in images], {"type": "text", "text": prompt}]\n            messages = [{"role": "user", "content": content}]\n            batch = processor.apply_chat_template(\n                messages,\n                tokenize=True,\n                add_generation_prompt=True,\n                return_dict=True,\n                return_tensors="pt",\n            ).to(device)\n            prompt_length = batch["input_ids"].shape[1]\n            with torch.inference_mode():\n                generated = model.generate(\n                    **batch,\n                    max_new_tokens=args.max_new_tokens,\n                    do_sample=False,\n                    num_beams=1,\n                    use_cache=True,\n                )\n            generated_only = generated[:, prompt_length:]\n            prediction = processor.batch_decode(generated_only, skip_special_tokens=True)[0]\n            reference = row["conversations"][1]["value"]\n            pred_answer, has_tag = answer_block(prediction)\n            ref_answer, _ = answer_block(reference)\n            exact = float(normalize(pred_answer) == normalize(ref_answer))\n            f1 = token_f1(pred_answer, ref_answer)\n\n            local["successes"] += 1\n            local["answer_tag"] += int(has_tag)\n            local["exact"] += exact\n            local["token_f1"] += f1\n            local["generated_tokens"] += int(generated_only.numel())\n            local[f"task_count::{task}"] += 1\n            prefix = f"task::{task}::"\n            local[prefix + "answer_tag"] += int(has_tag)\n            local[prefix + "exact"] += exact\n            local[prefix + "token_f1"] += f1\n\n            pred_json = parse_json_answer(prediction) if task in JSON_TASKS else None\n            ref_json = parse_json_answer(reference) if task in JSON_TASKS else None\n            if task in JSON_TASKS:\n                local["json_examples"] += 1\n                local[prefix + "json_examples"] += 1\n                valid = int(pred_json is not None)\n                local["valid_json"] += valid\n                local[prefix + "valid_json"] += valid\n            if task in {"ingredient_recognition", "full_food_analysis"}:\n                add_ingredient_scores(local, ingredient_names(pred_json), ingredient_names(ref_json), prefix)\n            if task in {"nutrition_estimation", "full_food_analysis"}:\n                add_numeric_scores(local, pred_json if isinstance(pred_json, dict) else None, ref_json if isinstance(ref_json, dict) else None, prefix)\n\n            if len(predictions) < 100:\n                predictions.append({\n                    "id": row.get("id"), "task": task, "prediction": prediction,\n                    "reference": reference, "exact": bool(exact), "token_f1": f1,\n                    "latency_seconds": time.perf_counter() - sample_started,\n                })\n        except Exception as exc:\n            local["failures"] += 1\n            if len(failures) < 50:\n                failures.append({\n                    "id": row.get("id"), "task": task,\n                    "error_type": type(exc).__name__, "error": str(exc),\n                    "traceback": traceback.format_exc(limit=5),\n                })\n            print(f"[EVAL][rank {rank}][ERROR] id={row.get(\'id\')} {type(exc).__name__}: {exc}", flush=True)\n        finally:\n            for image in locals().get("images", []):\n                try:\n                    image.close()\n                except Exception:\n                    pass\n            images = []\n        if index % args.progress_every == 0 or index == len(rows):\n            elapsed = time.perf_counter() - started\n            rate = index / max(elapsed, 1e-9)\n            print(\n                f"[EVAL][rank {rank}] {index:,}/{len(rows):,} rows | {rate:.2f} rows/s | "\n                f"failures={int(local[\'failures\'])} | gpu_peak={torch.cuda.max_memory_allocated(device)/2**30:.2f} GiB",\n                flush=True,\n            )\n\n    elapsed = time.perf_counter() - started\n    local["elapsed"] = elapsed\n    local["max_elapsed"] = elapsed\n    shard = {\n        "scores": dict(local),\n        "predictions": predictions,\n        "failures": failures,\n        "rank": rank,\n        "gpu_peak_allocated_gib": torch.cuda.max_memory_allocated(device) / 2**30,\n        "gpu_peak_reserved_gib": torch.cuda.max_memory_reserved(device) / 2**30,\n    }\n    gathered = [None for _ in range(world)]\n    dist.all_gather_object(gathered, shard)\n\n    if rank == 0:\n        total: defaultdict[str, float] = defaultdict(float)\n        total_predictions = []\n        total_failures = []\n        gpu_stats = []\n        for item in gathered:\n            for key, value in item["scores"].items():\n                if key == "max_elapsed":\n                    total[key] = max(total[key], value)\n                else:\n                    total[key] += value\n            total_predictions.extend(item["predictions"])\n            total_failures.extend(item["failures"])\n            gpu_stats.append({\n                "rank": item["rank"],\n                "peak_allocated_gib": item["gpu_peak_allocated_gib"],\n                "peak_reserved_gib": item["gpu_peak_reserved_gib"],\n            })\n        report = summarize(dict(total))\n        report.update({\n            "model": args.model,\n            "adapter": str(Path(args.adapter).resolve()),\n            "world_size": world,\n            "seed": args.seed,\n            "deterministic_generation": True,\n            "max_new_tokens": args.max_new_tokens,\n            "gpu_memory": gpu_stats,\n            "sample_predictions": total_predictions[:200],\n            "sample_failures": total_failures[:100],\n        })\n        output_path = Path(args.output)\n        output_path.parent.mkdir(parents=True, exist_ok=True)\n        output_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")\n        print("[EVAL][rank 0] FINAL TASK-AWARE REPORT", flush=True)\n        print(json.dumps({k: v for k, v in report.items() if not k.startswith("sample_")}, indent=2), flush=True)\n\n    if archive is not None:\n        archive.close()\n    dist.barrier()\n    dist.destroy_process_group()\n\n\nif __name__ == "__main__":\n    main()\n'
    evaluator_patches = [
        (
            'from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration',
            'from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig',
        ),
        (
            '    parser.add_argument("--max-new-tokens", type=int, default=256)\n',
            '    parser.add_argument("--max-new-tokens", type=int, default=256)\n'
            '    parser.add_argument("--load-in-4bit", type=lambda value: value.lower() == "true", default=True)\n'
            '    parser.add_argument("--quant-type", default="nf4")\n'
            '    parser.add_argument("--double-quant", type=lambda value: value.lower() == "true", default=True)\n'
            '    parser.add_argument("--compute-dtype", choices=("float16", "bfloat16", "float32"), default="float16")\n'
            '    parser.add_argument("--do-sample", type=lambda value: value.lower() == "true", default=False)\n'
            '    parser.add_argument("--temperature", type=float, default=0.0)\n'
            '    parser.add_argument("--num-beams", type=int, default=1)\n',
        ),
        (
            '    quantization = BitsAndBytesConfig(\n'
            '        load_in_4bit=True,\n'
            '        bnb_4bit_quant_type="nf4",\n'
            '        bnb_4bit_use_double_quant=True,\n'
            '        bnb_4bit_compute_dtype=torch.float16,\n'
            '    )\n',
            '    compute_dtype = getattr(torch, args.compute_dtype)\n'
            '    quantization = BitsAndBytesConfig(\n'
            '        load_in_4bit=args.load_in_4bit,\n'
            '        bnb_4bit_quant_type=args.quant_type,\n'
            '        bnb_4bit_use_double_quant=args.double_quant,\n'
            '        bnb_4bit_compute_dtype=compute_dtype,\n'
            '    )\n',
        ),
        ('    model = Qwen3VLForConditionalGeneration.from_pretrained(\n', '    model = AutoModelForImageTextToText.from_pretrained(\n'),
        ('        torch_dtype=torch.float16,\n', '        dtype=compute_dtype,\n'),
        (
            '                generated = model.generate(\n'
            '                    **batch,\n'
            '                    max_new_tokens=args.max_new_tokens,\n'
            '                    do_sample=False,\n'
            '                    num_beams=1,\n'
            '                    use_cache=True,\n'
            '                )\n',
            '                generation_kwargs = {\n'
            '                    "max_new_tokens": args.max_new_tokens,\n'
            '                    "do_sample": args.do_sample,\n'
            '                    "num_beams": args.num_beams,\n'
            '                    "use_cache": True,\n'
            '                }\n'
            '                if args.do_sample:\n'
            '                    generation_kwargs["temperature"] = args.temperature\n'
            '                generated = model.generate(**batch, **generation_kwargs)\n',
        ),
        (
            '            "deterministic_generation": True,\n'
            '            "max_new_tokens": args.max_new_tokens,\n',
            '            "deterministic_generation": not args.do_sample,\n'
            '            "max_new_tokens": args.max_new_tokens,\n'
            '            "decoding": {"do_sample": args.do_sample, "temperature": args.temperature, "num_beams": args.num_beams},\n'
            '            "quantization": {"load_in_4bit": args.load_in_4bit, "type": args.quant_type, "double_quant": args.double_quant, "compute_dtype": args.compute_dtype},\n',
        ),
    ]
    for old, new in evaluator_patches:
        assert EVALUATOR_SOURCE.count(old) == 1, f"Evaluator patch anchor changed or is ambiguous: {old[:100]!r}"
        EVALUATOR_SOURCE = EVALUATOR_SOURCE.replace(old, new, 1)
    print(f"[EVALUATOR PATCH] applied {len(evaluator_patches)} configuration/architecture patches")
    EVALUATOR_SCRIPT = WORK / "evaluate_foodr1_advanced.py"
    EVALUATOR_SCRIPT.write_text(EVALUATOR_SOURCE, encoding="utf-8")
    EVALUATOR_SCRIPT.chmod(0o755)
    subprocess.run([sys.executable, "-m", "py_compile", str(EVALUATOR_SCRIPT)], check=True)
    print(f"[EVALUATOR] path={EVALUATOR_SCRIPT}")
    print(f"[EVALUATOR] lines={len(EVALUATOR_SOURCE.splitlines()):,}")
    print(f"[EVALUATOR] sha256={sha256_file(EVALUATOR_SCRIPT, progress_gib=100)}")
    print("[EVALUATOR] Metrics: format compliance, JSON validity, classification F1/exact, ingredient-set scores, nutrition MAE/RMSE/MAPE.")



In [ ]:
# pyright: ignore
# 22 — Run deterministic generation on two GPUs and preserve every console line.
with stage("22", "Evaluate the final adapter on the untouched test split"):
    metrics_path = OUTPUT_DIR / "held_out_test_metrics.json"
    evaluation_command = [
        "torchrun", "--standalone", f"--nproc_per_node={WORLD_SIZE}", str(EVALUATOR_SCRIPT),
        "--test-json", str(SFT_DIR / "test.json"),
        "--adapter", str(OUTPUT_DIR),
        "--model", MODEL_ID,
        "--max-samples", str(MAX_TEST_SAMPLES),
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--load-in-4bit", cli_bool(LOAD_IN_4BIT),
        "--quant-type", BNB_4BIT_QUANT_TYPE,
        "--double-quant", cli_bool(BNB_4BIT_USE_DOUBLE_QUANT),
        "--compute-dtype", BNB_4BIT_COMPUTE_DTYPE,
        "--do-sample", cli_bool(GENERATION_DO_SAMPLE),
        "--temperature", str(GENERATION_TEMPERATURE),
        "--num-beams", str(GENERATION_NUM_BEAMS),
        "--min-pixels", str(MIN_IMAGE_PIXELS),
        "--max-pixels", str(MAX_IMAGE_PIXELS),
        "--seed", str(SEED),
        "--progress-every", "50",
        "--output", str(metrics_path),
    ]
    if DATA_MODE == "archive":
        evaluation_command.extend(["--archive", str(ARCHIVE)])
    else:
        evaluation_command.extend(["--dataset-root", str(DATASET_ROOT)])
    expected_evaluation_contract = {
        "--max-samples": MAX_TEST_SAMPLES, "--max-new-tokens": MAX_NEW_TOKENS,
        "--load-in-4bit": cli_bool(LOAD_IN_4BIT), "--quant-type": BNB_4BIT_QUANT_TYPE,
        "--double-quant": cli_bool(BNB_4BIT_USE_DOUBLE_QUANT), "--compute-dtype": BNB_4BIT_COMPUTE_DTYPE,
        "--do-sample": cli_bool(GENERATION_DO_SAMPLE), "--temperature": GENERATION_TEMPERATURE,
        "--num-beams": GENERATION_NUM_BEAMS, "--min-pixels": MIN_IMAGE_PIXELS,
        "--max-pixels": MAX_IMAGE_PIXELS, "--seed": SEED,
    }
    for option, expected in expected_evaluation_contract.items():
        actual = command_value(evaluation_command, option)
        assert actual == str(expected), f"Evaluation configuration drift for {option}: expected {expected!r}, got {actual!r}"
    assert evaluation_command[2] == f"--nproc_per_node={WORLD_SIZE}"
    print("[CONFIG PROPAGATION] Quantization, pixels, test coverage, and decoding settings reached the evaluator command.")
    (OUTPUT_DIR / "held_out_evaluation_command.json").write_text(json.dumps(evaluation_command, indent=2), encoding="utf-8")
    eval_cache_root = Path("/tmp/thrivelens_eval_hf_cache")
    eval_cache_root.mkdir(parents=True, exist_ok=True)
    eval_env = {
        **training_env,
        "HF_HOME": str(eval_cache_root),
        "HF_HUB_CACHE": str(eval_cache_root / "hub"),
        "TRANSFORMERS_CACHE": str(eval_cache_root / "transformers"),
        "XDG_CACHE_HOME": str(eval_cache_root / "xdg"),
    }
    print("[EVAL DISK] The held-out evaluator re-downloads the base model into the RAM-backed /tmp scratch cache; the 20 GiB working quota stays free for checkpoints and artifacts.")
    subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], check=True)
    with live_telemetry(TELEMETRY_SECONDS):
        run_streaming(
            evaluation_command,
            env=eval_env,
            log_path=OUTPUT_DIR / "held_out_evaluation.log",
            label="THRIVELENS-EVAL",
        )
    assert metrics_path.exists(), "Distributed evaluator did not produce its report"
    held_out_metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    assert held_out_metrics["world_size"] == WORLD_SIZE
    assert held_out_metrics["deterministic_generation"] == (not GENERATION_DO_SAMPLE)
    assert held_out_metrics["decoding"]["num_beams"] == GENERATION_NUM_BEAMS
    assert held_out_metrics["quantization"]["compute_dtype"] == BNB_4BIT_COMPUTE_DTYPE
    assert held_out_metrics["successful_samples"] > 0
    assert held_out_metrics["failure_rate"] <= 0.01, (
        f"Evaluation failure rate {held_out_metrics['failure_rate']:.2%} exceeds the 1% integrity threshold"
    )
    print_json({key: value for key, value in held_out_metrics.items() if not key.startswith("sample_")})


In [ ]:
# pyright: ignore
# 23 — Produce human-readable task tables, failure samples, and a model-quality report.
with stage("23", "Summarize held-out quality and hallucination-related indicators"):
    import pandas as pd

    task_rows = []
    for task, values in held_out_metrics["by_task"].items():
        row = {
            "task": task,
            "samples": values.get("samples"),
            "exact_match": values.get("exact_match"),
            "token_f1": values.get("token_f1"),
            "answer_tag_compliance": values.get("answer_tag_compliance"),
            "valid_json_rate": values.get("valid_json_rate"),
            "ingredient_set_precision": values.get("ingredient_set_precision"),
            "ingredient_set_recall": values.get("ingredient_set_recall"),
            "ingredient_set_f1": values.get("ingredient_set_f1"),
        }
        for field, numeric in values.get("numeric_metrics", {}).items():
            row[f"{field}_mae"] = numeric.get("mae")
            row[f"{field}_rmse"] = numeric.get("rmse")
            row[f"{field}_mape_percent"] = numeric.get("mape_percent")
        task_rows.append(row)
    task_frame = pd.DataFrame(task_rows).sort_values("task")
    task_csv = OUTPUT_DIR / "held_out_task_metrics.csv"
    task_frame.to_csv(task_csv, index=False)
    print(task_frame.to_string(index=False))

    predictions_path = OUTPUT_DIR / "held_out_sample_predictions.jsonl"
    with predictions_path.open("w", encoding="utf-8") as handle:
        for item in held_out_metrics.get("sample_predictions", []):
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")
    failures_path = OUTPUT_DIR / "held_out_sample_failures.jsonl"
    with failures_path.open("w", encoding="utf-8") as handle:
        for item in held_out_metrics.get("sample_failures", []):
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")

    report_lines = [
        "# ThriveLens Food-R1 held-out quality report",
        "",
        f"- Evaluated: {held_out_metrics['successful_samples']:,} successful of {held_out_metrics['requested_samples']:,} requested records",
        f"- Deterministic generation: {held_out_metrics['deterministic_generation']}",
        f"- Two-GPU evaluation: world size {held_out_metrics['world_size']}",
        f"- Failure rate: {held_out_metrics['failure_rate']:.4%}",
        f"- Answer-tag compliance: {held_out_metrics['answer_tag_compliance']:.4%}",
        f"- JSON validity on structured tasks: {held_out_metrics['valid_json_rate_on_json_tasks']:.4%}",
        f"- Overall exact match: {held_out_metrics['exact_match']:.4%}",
        f"- Overall token F1: {held_out_metrics['token_f1']:.4%}",
        "",
        "These are measured hallucination-related indicators, not a claim that hallucination is impossible. Low format compliance, invalid JSON, low ingredient F1, or large nutrition errors must be treated as deployment blockers for the affected task.",
        "",
        "## Per-task metrics",
        "",
        task_frame.to_markdown(index=False),
        "",
        f"Sample generation failures retained: {len(held_out_metrics.get('sample_failures', []))}",
    ]
    quality_report_path = OUTPUT_DIR / "HELD_OUT_QUALITY_REPORT.md"
    quality_report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
    print(f"[REPORT] {quality_report_path}")
    if held_out_metrics.get("sample_failures"):
        print("[REPORT] First five evaluation failures:")
        print_json(held_out_metrics["sample_failures"][:5])



## Final delivery package

The final ZIP contains the deployable LoRA adapter, processor/tokenizer assets, complete run manifest, model and adapter summaries, diagnostics, metrics, logs, exact commands, and data/software audits. Intermediate `checkpoint-*` directories are intentionally excluded from the deployment ZIP; keep them separately only when you need to resume training. With the optional Hub auto-sync enabled, the newest checkpoint and the completed adapter are mirrored to your private repo automatically, so a fresh session can restore them without the manual re-attach step.

In [ ]:
# pyright: ignore
# 24 — Build a compact, checksummed deployment archive without bulky optimizer checkpoints.
with stage("24", "Package the trained adapter and every reproducibility/quality artifact"):
    import textwrap

    delivery_readme = textwrap.dedent(f'''    # ThriveLens Food-R1 QLoRA adapter

    Base model: `{MODEL_ID}`  
    Training: fixed {EPOCHS} epochs, dual-GPU DeepSpeed ZeRO-2, FP16 NF4 QLoRA  
    Early stopping: disabled  
    Dataset archive SHA-256: `{archive_sha256}`

    This is a parameter-efficient adapter, not a redistribution of the Food-R1 base weights. Load the public base model, then load this directory with `peft.PeftModel.from_pretrained`. Use deterministic decoding for evaluated production paths and validate the structured output before trusting nutrition values.

    The held-out report and task CSV provide the model's measured quality. Do not infer medical accuracy from image-only nutrition estimates.
    ''')
    (OUTPUT_DIR / "DELIVERY_README.md").write_text(delivery_readme, encoding="utf-8")

    delivery_path = Path("/kaggle/working/ThriveLens_FoodR1_Dual_T4_QLoRA_Final.zip")
    if delivery_path.exists():
        delivery_path.unlink()
    support_artifacts = [
        WORK / "requested_configuration.json",
        WORK / "dataset_audit.json",
        WORK / "image_decode_audit.json",
        WORK / "software_versions.json",
        WORK / "pip_freeze.txt",
        WORK / "model_architecture_summary.json",
        WORK / "input_length_audit.json",
        WORK / "smoke_model_parameter_summary.json",
        WORK / "dual_gpu_nccl_preflight.log",
        WORK / "dual_gpu_smoke_test.log",
        WORK / "full_training_command.json",
        WORK / "full_training_command.sh",
        WORK / "verified_training_argument_contract.json",
        WORK / "held_out_evaluation_command.json",
    ]
    packaged = []
    with zipfile.ZipFile(delivery_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as archive:
        for path in sorted(OUTPUT_DIR.rglob("*")):
            if not path.is_file() or any(part.startswith("checkpoint-") for part in path.relative_to(OUTPUT_DIR).parts):
                continue
            relative = Path("adapter_and_reports") / path.relative_to(OUTPUT_DIR)
            archive.write(path, relative.as_posix())
            packaged.append(relative.as_posix())
        for path in support_artifacts:
            if path.exists():
                relative = Path("reproducibility") / path.name
                archive.write(path, relative.as_posix())
                packaged.append(relative.as_posix())

    delivery_sha256 = sha256_file(delivery_path, progress_gib=0.25)
    checksum_path = delivery_path.with_suffix(".sha256")
    checksum_path.write_text(f"{delivery_sha256}  {delivery_path.name}\n", encoding="utf-8")
    package_report = {
        "archive": str(delivery_path),
        "size_mib": delivery_path.stat().st_size / 2**20,
        "sha256": delivery_sha256,
        "file_count": len(packaged),
        "checkpoints_excluded": True,
        "first_30_entries": packaged[:30],
    }
    print_json(package_report)
    print(f"[DOWNLOAD] Open Kaggle's Output panel and download: {delivery_path.name}")



In [ ]:
# pyright: ignore
# 25 — Final A* notebook completion checklist.
with stage("25", "Print the final delivery checklist"):
    final_checklist = {
        "dataset_records_loaded": TRAIN_ROWS_EXPECTED + VAL_ROWS_EXPECTED + TEST_ROWS_EXPECTED,
        "group_leakage": 0,
        "two_cuda_devices": torch.cuda.device_count() == WORLD_SIZE,
        "nccl_two_rank_preflight": True,
        "dual_gpu_optimizer_smoke_test": RUN_DUAL_GPU_SMOKE_TEST,
        "epochs_completed": completed_epoch,
        "early_stopping_used": EARLY_STOPPING,
        "configuration_propagation_verified": (OUTPUT_DIR / "verified_training_argument_contract.json").exists(),
        "vision_tower_frozen_for_4bit_safety": FREEZE_VISION_TOWER,
        "adapter_integrity_verified": True,
        "validation_losses_finite": True,
        "held_out_generation_world_size": held_out_metrics["world_size"],
        "held_out_successful_samples": held_out_metrics["successful_samples"],
        "held_out_failure_rate": held_out_metrics["failure_rate"],
        "delivery_zip": str(delivery_path),
        "delivery_sha256": delivery_sha256,
    }
    assert all([
        final_checklist["group_leakage"] == 0,
        final_checklist["two_cuda_devices"],
        final_checklist["epochs_completed"] >= EPOCHS,
        not final_checklist["early_stopping_used"],
        final_checklist["configuration_propagation_verified"],
        final_checklist["held_out_generation_world_size"] == WORLD_SIZE,
        Path(final_checklist["delivery_zip"]).exists(),
    ])
    print_json(final_checklist)
    print("\nTHRIVELENS FOOD-R1 TRAINING, VALIDATION, TESTING, REPORTING, AND PACKAGING COMPLETED.")
